# 🔆 8장 · 내일의 발전량: 회귀 분석 (v8판 실습 노트북)

『빅데이터분석 with 파이썬』 8장의 **05. 파이썬 실습**과 **미니 프로젝트**를 그대로 따라 하는 노트북입니다. 실습 번호는 책과 같습니다.

| 표시 | 뜻 | 실습 |
|---|---|---|
| ★ 필수 | 수업 시간에 모두 함께 | 실습 예제 8-1~8-8 |
| 작은 예제 | 알고리즘을 손 계산으로 | 실습 예제 8-9(VIF), 8-10(릿지·라소, 나무와 숲) |
| 코랩 실습 | 책에는 문제·결과만, 코드는 여기에만 | 8-1 경사하강법, 8-2 이슬점 VIF, 8-3 네 모델 견주기, 8-4 원자료 정리 |

**데이터**
- 골턴의 가족 키(1886년): GitHub 저장소 `vincentarelbundock/Rdatasets`에서 바로 읽습니다.
- 서울 태양광 발전량: 한국전력거래소 "지역별 시간별 태양광 및 풍력 발전량" 2025년(출처 한국전력거래소, 공공데이터포털, 공공누리 제1유형).
- 서울 날씨: 기상청 종관기상관측(ASOS) 서울(108) 시간 자료 2023-01-01~2026-10-17(출처 기상청, 기상자료개방포털, 공공누리 제1유형).
- 미니 프로젝트: 한국전력거래소 "시간별 전국 전력수요량" 2021~2025년(출처 한국전력거래소, 공공데이터포털, 공공누리 제1유형).

한국 데이터 세 파일은 **수업 자료로 받은 정리본(UTF-8 CSV)**을 씁니다. 코랩 왼쪽 파일 창에 올려 두면 되고, 다른 폴더에 두었다면 05절 첫 코드 셀의 `DATA = ""` 한 줄만 그 폴더로 고칩니다(예: `DATA = "/content/drive/MyDrive/data/"`). 포털에서 원자료를 직접 받아 정리하는 [코랩 실습 8-4]는 실습 예제 8-10 뒤에 있고, 원자료가 없으면 건너뜁니다.

각 실습은 책처럼 ❶❷❸ 단계로 나뉘어 있고, 단계마다 코드 셀 하나입니다. [직접 해 보기]는 빈 셀에 먼저 써 보고, 그다음 정답 셀을 실행하십시오(책에는 문제만 있고 정답은 이 노트북에만 있습니다). 책에서 줄인 단계(8-4의 ❶~❸, 8-5의 ❾, 8-8의 ❹)도 번호 그대로 여기에 있습니다.

코드 셀은 **위에서부터 차례로** 실행하십시오(`Shift + Enter`). 아래 셀은 위에서 만든 변수를 이어서 씁니다.

## 0 · 준비

그래프의 한글이 네모(□)로 깨지지 않게 해 주는 **koreanize-matplotlib**와 나라별 공휴일 목록 **holidays**(미니 프로젝트)를 설치합니다. 코랩에 이미 있으면 바로 넘어갑니다. 판다스, 스태츠모델, 사이킷런, 맷플롯립은 코랩에 들어 있습니다.

In [ ]:
!pip install -q koreanize-matplotlib holidays

**데이터 받기** (깃허브 저장소판에서 덧붙인 셀) — 아래 셀은 이 장이 쓰는 강의 자료 파일을 저장소의 `data` 폴더에서 코랩으로 내려받습니다.

In [ ]:
# 이 장의 강의 자료 파일을 책 깃허브 저장소에서 받아 옵니다.
# (파일을 코랩 왼쪽 📁 파일 창에 직접 끌어다 올렸다면 이 셀은 건너뛰어도 됩니다.)
import os, urllib.parse, urllib.request
BASE = "https://raw.githubusercontent.com/yeonu69/bigdata-python-book/main/ch08/data/"
for f in [
    "asos_서울108_시간자료_전체열_2023_2026.csv",
    "kpx_시간별_전국_전력수요량_2021_2025.csv",
    "kpx_지역별_시간별_태양광풍력_발전량_2025.csv",
]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(BASE + urllib.parse.quote(f), f)
    print("준비됨:", f, f"({os.path.getsize(f) / 1e6:.1f} MB)")

## 04절 실습 예제: 손 풀이를 코드로

#### [실습 예제 8-1] [예제 8-1]을 파이썬 코드로 구현하기

손으로 채운 [그림 8-17]의 표를 넘파이 배열 계산으로 한 번에 구합니다.

In [ ]:
import numpy as np
x = np.array([2, 4, 6, 8, 10])          # 공부시간
y = np.array([60, 70, 75, 85, 90])      # 점수
dx = x - x.mean()                       # ① x의 편차
dy = y - y.mean()                       #    y의 편차
b = (dx * dy).sum() / (dx ** 2).sum()   # ② 함께 움직임 ÷ x 흩어짐
a = y.mean() - b * x.mean()             # ③ 절편: 평균점을 지나게
print("평균:", x.mean(), y.mean())
print("편차 곱의 합:", (dx * dy).sum(), "/ x 편차 제곱합:",
      (dx ** 2).sum())
print(f"ŷ = {a:.1f} + {b:.2f}x")
sse = ((y - (a + b * x)) ** 2).sum()    # 직선이 남긴 흩어짐
sst = (dy ** 2).sum()                   # 처음 흩어짐
print(f"SSE {sse:.1f}, SST {sst:.0f}, R² {1 - sse / sst:.3f}")

- `x - x.mean()`: 배열의 모든 칸에서 평균을 한 번에 뺍니다. 표의 "x − x̄" 열이 됩니다.
- `(dx * dy).sum()`: 같은 자리끼리 곱해 더합니다. 표의 150입니다.
- `f"ŷ = {a:.1f} + {b:.2f}x"`: 계산한 값을 소수 자리를 정해 식 모양으로 찍습니다.

손 계산과 같은 직선이 나왔고, 4.3의 R² 0.987도 함께 확인됩니다. [실습 예제 8-4]에서는 이 계산을 판다스 표로 다시 해 보고 그래프로 그립니다.

#### [실습 예제 8-2] [예제 8-2]를 파이썬 코드로 구현하기

세 걸음을 함수 하나로 묶고, 여러 아버지 키를 넣어 평균으로의 회귀를 한눈에 봅니다.

In [ ]:
f_mean, f_sd = 175.6, 5.86              # 아버지 키 평균, 표준편차
s_mean, s_sd = 175.9, 6.66              # 아들 키 평균, 표준편차
r = 0.392                               # 상관계수
def predict_son(father):
    z_f = (father - f_mean) / f_sd      # ① 아버지 키의 z-점수
    z_s = r * z_f                       # ② r배로 줄인다
    return z_f, z_s, s_mean + z_s * s_sd   # ③ cm로 되돌린다
z_f, z_s, son = predict_son(185.7)
print(f"z {z_f:.2f} → {z_s:.2f}, 아들 키 예측 {son:.1f}cm")
for father in [160, 170, 180, 190]:     # 여러 아버지 키로
    print(f"아버지 {father}cm → 아들 {predict_son(father)[2]:.1f}cm")

- `def predict_son(father):`: 세 걸음을 함수로 묶습니다. 아버지 키만 바꿔 넣으면 됩니다.
- `return z_f, z_s, ...`: 값 셋을 한꺼번에 돌려줍니다. `predict_son(190)[2]`는 그 가운데 세 번째(아들 키)입니다.

아들 키는 손 계산과 같은 180.4cm입니다. 가운데 값이 0.68인 것은 코드가 1.72로 반올림하지 않은 1.7235를 그대로 곱했기 때문입니다. 아버지 키가 160cm에서 190cm로 30cm 벌어질 때 아들 키는 168.9cm에서 182.3cm로 13.4cm만 벌어집니다. 양쪽 끝이 모두 평균(175.9cm) 쪽으로 당겨졌습니다.

## 05. 파이썬 실습: 키에서 서울의 발전량까지

실습은 코랩(Colab)에서 노트북 "8장_회귀분석_실습_colab_v6.ipynb"를 위에서부터 차례로 실행합니다. 책의 코드는 노트북과 한 글자도 다르지 않고, 책의 [실행 결과]와 같은 값이 나오는지 견주어 보십시오. 골턴의 키는 인터넷 주소에서 바로 읽습니다.

> **[STEP] 실습 전에 미리 살펴보기**
>
> 어떤 데이터를 쓰나요? 골턴의 키(인터넷 주소에서 바로 읽음), 그리고 수업 자료로 받은 한국 데이터 세 파일(태양광 발전량, 서울 날씨, 전국 전력수요)입니다. 세 파일은 코랩 왼쪽 파일 창에 올려 둡니다.
>
> 어떤 도구를 쓰나요? 표는 판다스(pandas), 숫자 계산은 넘파이(numpy), 그래프는 매트플롯립(matplotlib), 회귀 모델은 스태츠모델(statsmodels)과 사이킷런(scikit-learn)을 씁니다.
>
> 무엇을 하나요? 8-3·8-4는 01·04절의 개념을 코드로 확인하고(8-1·8-2는 04절의 예제 풀이 코드), 8-5~8-8은 03절의 분석을 "잇기 → 빈칸 → 직선·기준선 → 변수 더하기 → 범위" 순서로 따라 합니다. 8-9·8-10은 VIF, 릿지·라소, 나무와 숲을 작은 예제로 손 계산합니다. 서울 데이터에 쓰는 긴 코드는 [코랩 실습 8-1~8-4]로 노트북에만 있습니다.
>
> 어떻게 읽나요? 단계마다 할 일 → 코드 → 한 줄씩 읽기 → 실행 결과 → 결과 읽기 순서입니다. 실습 끝의 [직접 해 보기]는 코드 한두 줄을 바꿔 보는 문제이고, 정답 코드와 결과를 함께 실었습니다. 자주 막히는 곳은 [자주 하는 실수] 상자에 모았습니다.

**❶ 도구 불러오기**

모든 실습의 첫 셀입니다. 한 번만 실행하면 노트북을 닫을 때까지 유지됩니다.

In [ ]:
import numpy as np                     # 숫자 계산
import pandas as pd                    # 표(데이터프레임)
import matplotlib.pyplot as plt        # 그래프
# 코랩에서 한 번: !pip install koreanize-matplotlib
import koreanize_matplotlib            # 그래프의 한글 글꼴
DATA = ""                              # 데이터 폴더(파일 창이면 빈칸)
print(pd.__version__, np.__version__)

- `import pandas as pd`: 판다스를 `pd`라는 짧은 이름으로 부릅니다. 이후 `pd.read_csv( )`처럼 씁니다. `np`, `plt`도 모두가 쓰는 약속된 별명입니다.
- `koreanize_matplotlib`: 그래프에 한글이 네모(□)로 깨지지 않게 글꼴을 맞춥니다. 코랩에는 처음에 없으니 `!pip install`로 한 번 설치합니다.
- `DATA = ""`: 데이터 파일이 있는 폴더 이름입니다. 파일을 "drive/MyDrive/data/"에 두었다면 그 글자를 넣습니다. 끝의 `/`를 잊지 마세요.

판다스와 넘파이의 버전입니다. 코랩의 버전은 이와 다를 수 있습니다. 오류가 나면 먼저 버전부터 확인합니다.

> **[자주 하는 실수] 셀을 건너뛰고 실행하기**
>
> 노트북은 위에서 아래로 차례로 실행해야 합니다. 셀 하나를 건너뛰면 `NameError: name 'sons' is not defined`처럼 "그런 이름이 없다"는 오류가 납니다. 코랩을 다시 열었거나 오래 쉬어 연결이 끊겼을 때도 같습니다. 이때는 메뉴의 "런타임 → 이전 셀 모두 실행"을 누릅니다.
>
> 파일을 못 찾는다는 `FileNotFoundError`는 파일 창에 파일을 올리지 않았거나, `DATA` 폴더 이름이 틀렸거나, 파일 이름의 한 글자(밑줄, 연도)가 다를 때 납니다. 파일 창에서 파일 이름 옆의 점 세 개 → "경로 복사"를 눌러 붙여 넣으면 틀리지 않습니다.

#### [실습 예제 8-3 ★] 골턴의 키와 평균으로의 회귀

골턴의 키 데이터에서 키 큰 아버지와 키 작은 아버지의 아들 키를 견주고, 회귀선을 구해 그리시오(01절).

**❶ 데이터 읽고 생김새 보기**

데이터를 받으면 가장 먼저 몇 줄·몇 열인지, 앞의 몇 줄이 어떻게 생겼는지 봅니다.

In [ ]:
URL = ("https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/"
       "master/csv/HistData/GaltonFamilies.csv")
galton = pd.read_csv(URL)              # 인터넷 주소의 CSV를 표로 읽는다
print(galton.shape)                    # (줄 수, 열 수)
cols = ["father", "mother", "gender", "childHeight"]
print(galton[cols].head(3))            # 네 열의 앞 세 줄

- `pd.read_csv(URL)`: 쉼표로 구분된 글자 파일(CSV)을 표(데이터프레임)로 읽습니다. 컴퓨터의 파일 이름 대신 인터넷 주소를 넣어도 됩니다.
- 주소를 괄호 안에서 두 줄로 나눈 것은 긴 글자를 보기 좋게 쓰려는 것입니다. 파이썬은 괄호 안에 나란히 놓인 글자를 하나로 잇습니다.
- `galton.shape`: (줄 수, 열 수)입니다. 괄호가 없는 것은 계산이 아니라 표가 가진 정보이기 때문입니다.
- `galton[cols].head(3)`: 열 이름 목록으로 열을 고르고, 앞 세 줄만 봅니다.

934줄은 자녀 934명, 한 줄이 자녀 한 명입니다. 키는 인치이고, 첫 가족(아버지 78.5인치)의 자녀가 세 줄에 걸쳐 나옵니다. 왼쪽의 0, 1, 2는 판다스가 붙인 줄 번호(인덱스)입니다.

> **[개념 노트] 데이터프레임: 줄은 관측, 열은 변수**
>
> 판다스의 **데이터프레임**(DataFrame)은 엑셀 시트 같은 표입니다. 한 줄은 관측 하나(아들 한 명, 한 시각), 한 열은 변수 하나(키, 일사)입니다. 이 "줄 = 관측, 열 = 변수" 모양을 깔끔한 데이터(tidy data)라고 하며, 회귀를 비롯한 거의 모든 모델이 이 모양을 받습니다.
>
> 자주 쓰는 동작은 네 가지입니다. 열 고르기 `df["키"]`, 조건으로 줄 고르기 `df[df["키"] > 180]`(불리언 인덱싱), 새 열 만들기 `df["cm"] = df["inch"] * 2.54`, 요약 보기 `df.describe( )`입니다. 조건을 여러 개 걸 때는 `&`(그리고), `|`(또는)로 잇고 조건마다 괄호를 씌웁니다. 데이터를 받으면 먼저 `df.shape`(줄·열 수), `df.head( )`(앞 다섯 줄), `df.info( )`(열 이름·자료형·빈칸)로 생김새부터 확인하는 습관을 들이세요.

**❷ 열 이름과 성별 세기**

In [ ]:
print(galton.columns.tolist())
print(galton["gender"].value_counts())  # 남녀 각각 몇 명

- `galton.columns.tolist()`: 열 이름을 목록으로 봅니다. 이후 코드에서 열 이름을 한 글자도 틀리지 않게 쓰려면 여기서 확인합니다.
- `value_counts()`: 열에 나오는 값마다 개수를 셉니다. 범주형 열을 처음 볼 때 늘 씁니다.

아들 481명, 딸 453명입니다. 1.1의 "아들 481명"이 여기서 나옵니다. `midparentHeight`는 부모 키의 평균에 가까운 값으로, 골턴이 만든 열입니다.

**❸ 아들만 고르고 cm로 바꾸기**

In [ ]:
sons = galton[galton["gender"] == "male"].copy()   # 아들만
sons["아버지"] = sons["father"] * 2.54              # 인치 → cm
sons["아들"] = sons["childHeight"] * 2.54
print(len(sons), "명")
print(sons[["아버지", "아들"]].describe().round(1))

- `galton["gender"] == "male"`: 줄마다 "아들인가?"를 묻고 참·거짓 934개를 돌려줍니다. 이것을 대괄호에 넣으면 참인 줄만 남습니다(불리언 인덱싱).
- `.copy()`: 골라낸 표를 따로 떼어 냅니다. 빼면 다음 줄에서 열을 더할 때 `SettingWithCopyWarning` 경고가 납니다.
- `sons["father"] * 2.54`: 열 전체에 2.54를 한 번에 곱해 새 열 "아버지"를 만듭니다. for 문이 필요 없습니다.
- `describe()`: 개수, 평균, 표준편차, 최솟값, 사분위수, 최댓값을 한꺼번에 보여 줍니다. `.round(1)`은 소수 첫째 자리까지 반올림합니다.

아버지 평균 175.6cm, 아들 평균 175.9cm로 두 세대의 평균은 거의 같고, 흩어짐(표준편차)도 5.9와 6.7로 비슷합니다. 평균과 흩어짐이 그대로인데 키 큰 아버지의 아들만 평균 쪽으로 온다는 것이 평균으로의 회귀의 묘한 점입니다.

**❹ 위아래 10% 견주기**

In [ ]:
q90 = sons["아버지"].quantile(0.9)     # 아버지 키의 90% 자리 값
q10 = sons["아버지"].quantile(0.1)     # 10% 자리 값
tall = sons[sons["아버지"] >= q90]     # 키 큰 아버지 상위 10%
short = sons[sons["아버지"] <= q10]    # 키 작은 아버지 하위 10%
print(f"기준: {q10:.1f}cm 이하, {q90:.1f}cm 이상")
for name, g in [("키 큰", tall), ("키 작은", short)]:
    print(f"{name} 아버지 {len(g)}명: {g['아버지'].mean():.1f}cm"
          f" → 아들 {g['아들'].mean():.1f}cm")

- `quantile(0.9)`: 작은 쪽부터 줄 세웠을 때 90% 자리에 있는 값(90% 분위수)입니다. 그보다 크거나 같은 줄이 상위 10%입니다.
- `f"…{q10:.1f}…"`: 글자 안의 `{ }`에 변수의 값을 넣는 f-문자열입니다. `:.1f`는 소수 첫째 자리까지 보이라는 뜻입니다.
- `for name, g in [(…), (…)]`: 이름과 표를 짝지은 목록을 하나씩 꺼내, 같은 출력 코드를 두 번 씁니다.

키 큰 아버지(평균 185.7cm)의 아들은 180.4cm로 5.3cm, 키 작은 아버지(165.9cm)의 아들은 171.6cm로 5.7cm만큼 아버지보다 평균 쪽에 있습니다.

**❺ 거꾸로 보기: 키 큰 아들의 아버지**

In [ ]:
tall_son = sons[sons["아들"] >= sons["아들"].quantile(0.9)]
print(f"키 큰 아들 {len(tall_son)}명: {tall_son['아들'].mean():.1f}cm"
      f" → 그 아버지 {tall_son['아버지'].mean():.1f}cm")

키 큰 아들 49명(187.1cm)의 아버지는 180.5cm입니다. 아버지 쪽으로 봐도 평균으로 "돌아왔으니", 회귀는 아버지가 아들에게 미치는 힘이 아니라 우연이 만든 현상입니다(01절 1.4).

**❻ 회귀선과 상관계수 구하기**

In [ ]:
b, a = np.polyfit(sons["아버지"], sons["아들"], 1)   # 기울기, 절편
r = sons["아버지"].corr(sons["아들"])                 # 상관계수
print(f"아들 = {a:.1f} + {b:.3f} × 아버지")
print(f"상관계수 {r:.3f}, R² {r ** 2:.3f}")
print(f"아버지 185cm → 아들 {a + b * 185:.1f}cm")

- `np.polyfit(x, y, 1)`: x와 y에 맞는 1차식(직선)을 최소제곱법으로 찾아 [기울기, 절편] 순서로 돌려줍니다. 그래서 `b, a =`로 두 변수에 나눠 받습니다. 마지막 1을 2로 바꾸면 2차식(곡선)을 찾습니다.
- `.corr( )`: 두 열의 상관계수 r입니다. 단순 회귀에서는 r을 제곱하면 R²입니다.
- `a + b * 185`: 구한 직선으로 아버지 185cm의 아들 키를 예측합니다.

기울기 0.447은 "아버지가 1cm 크면 아들은 평균 0.447cm 크다"는 뜻입니다. R² 0.154는 아들 키 흩어짐의 15%만 아버지 키로 설명된다는 뜻이고, 나머지 85%는 어머니의 키, 영양, 우연 같은 다른 몫입니다.

**❼ 산점도와 회귀선 그리기**

In [ ]:
plt.figure(figsize=(5, 4.4))
plt.scatter(sons["아버지"], sons["아들"], s=8, alpha=0.4, color="gray")
xs = np.array([155, 200])                             # 직선의 양 끝
plt.plot(xs, a + b * xs, color="#C8234B", label="회귀선")
plt.plot(xs, xs, "k--", lw=1, label="아들 = 아버지")   # 기울기 1인 점선
plt.xlabel("아버지 키(cm)")
plt.ylabel("아들 키(cm)")
plt.legend()
plt.show()

- `plt.figure(figsize=(5, 4.4))`: 그림의 가로·세로 크기(인치)를 정합니다.
- `plt.scatter(x, y, s=8, alpha=0.4)`: 점을 찍습니다. `s`는 점의 크기, `alpha=0.4`는 반투명이라 겹친 곳이 진하게 보입니다.
- `xs = np.array([155, 200])`: 직선은 양 끝 두 점만 알면 그을 수 있습니다. `a + b * xs`가 두 점의 높이입니다.
- `"k--"`: 검은색(k) 점선(--)입니다. 기울기 1인 "아들 = 아버지" 선을 견줄 잣대로 긋습니다.
- `label=`과 `plt.legend()`: 선마다 이름을 붙이고 범례 상자에 모아 보여 줍니다.

**결과 읽기.** 빨간 회귀선(기울기 0.447)이 점선(기울기 1)보다 누워 있는 것이 평균으로의 회귀를 그림으로 본 모습입니다. 점선 오른쪽 위에서는 대부분의 점이 점선 아래에, 왼쪽 아래에서는 위에 있습니다. 점들이 넓게 퍼져 있으니 R²가 낮은 것도 눈으로 확인됩니다.

> **[개념 노트] 숫자보다 그래프를 먼저**
>
> 회귀 분석에서 가장 먼저 그리는 그래프는 **산점도**입니다. 직선이 어울리는지, 휘었는지, 혼자 떨어진 점(이상값)이 있는지가 한눈에 보입니다. 상관계수나 R²가 같아도 점의 모양은 전혀 다를 수 있습니다(앤스컴의 네 데이터). 매트플롯립에서는 `plt.scatter( )`(점), `plt.plot( )`(선), `plt.hist( )`(히스토그램), `plt.bar( )`(막대) 네 함수면 이 장의 그래프를 모두 그릴 수 있습니다. 축 이름에는 단위까지 적습니다.

**[직접 해 보기]** 딸과 어머니로 같은 회귀를 해 보시오. 기울기는 0.447보다 클까요, 작을까요?

In [ ]:
# 여기에 직접 코드를 써 보십시오.

**정답 코드** (직접 해 본 뒤에 실행해 보십시오)

In [ ]:
dau = galton[galton["gender"] == "female"].copy()
dau["어머니"] = dau["mother"] * 2.54
dau["딸"] = dau["childHeight"] * 2.54
b2, a2 = np.polyfit(dau["어머니"], dau["딸"], 1)
r2 = dau["어머니"].corr(dau["딸"])
print(f"딸 = {a2:.1f} + {b2:.3f} × 어머니, 상관계수 {r2:.3f}")

딸도 어머니보다 평균 쪽으로 돌아옵니다(기울기 0.318). 아들과 아버지보다 상관계수가 더 낮아, 회귀가 더 강하게 나타납니다.

> **[자주 하는 실수] 열 이름과 대괄호**
>
> `sons["아버지 "]`처럼 열 이름에 빈칸 하나만 더 들어가도 `KeyError: '아버지 '`가 납니다. 오류 메시지 마지막 줄의 따옴표 안을 보면 무엇이 틀렸는지 보입니다. 열 하나는 `df["키"]`, 여러 열은 `df[["키", "몸무게"]]`처럼 대괄호를 두 겹으로 씁니다. 한 겹에 쉼표로 쓰면 `KeyError`가 납니다.
>
> 조건을 두 개 이상 걸 때는 `and` 대신 `&`를 쓰고 조건마다 괄호를 씌웁니다. `df[df["a"] > 1 and df["b"] > 2]`는 `ValueError: The truth value of a Series is ambiguous` 오류가 납니다.

#### [실습 예제 8-4 ★] 최소제곱 직선을 손 계산처럼 구하기

[예제 8-1]의 계산을 (식 8-3)과 (식 8-4) 그대로 코드로 옮기고, 기울기를 바꾸면 잔차 제곱합이 어떻게 변하는지 그리시오(04절).

**❶ 데이터 만들기**

In [ ]:
study = pd.DataFrame({"공부시간": [2, 4, 6, 8, 10],
                      "점수": [60, 70, 75, 85, 90]})
x, y = study["공부시간"], study["점수"]
print(study)
print("평균:", x.mean(), y.mean())

- `pd.DataFrame({열 이름: [값들], …})`: 사전의 키가 열 이름, 리스트가 그 열의 값이 됩니다. 손으로 만든 작은 데이터로 계산을 확인할 때 씁니다.
- `x, y = study["공부시간"], study["점수"]`: 두 열을 짧은 이름으로 받아 식처럼 쓰려는 것입니다.

**❷ 편차와 그 곱을 표로 보기**

(식 8-3)의 분자와 분모를 한 칸씩 계산해 표에 열로 붙입니다. [그림 8-18]의 표가 그대로 나옵니다.

In [ ]:
study["x편차"] = x - x.mean()
study["y편차"] = y - y.mean()
study["편차곱"] = study["x편차"] * study["y편차"]
study["x편차²"] = study["x편차"] ** 2
print(study)
print("합:", study["편차곱"].sum(), study["x편차²"].sum())

- `x - x.mean()`: 다섯 값 모두에서 평균 6을 한 번에 뺍니다(벡터 연산). 결과는 다섯 칸짜리 열입니다.
- `** 2`: 거듭제곱입니다. `^`가 아니라 `**`를 씁니다.
- `.sum()`: 식의 Σ(모두 더하라)입니다.

편차곱의 합 150은 x와 y가 함께 움직인 정도, x편차²의 합 40은 x가 흩어진 정도입니다. 세 번째 학생은 x가 평균이라 두 값 모두 0입니다(`-0.0`은 컴퓨터가 0을 쓰는 방식일 뿐 0과 같습니다).

> **[개념 노트] 반복문 없이 열 전체를 계산하는 벡터 연산**
>
> `x - x.mean()`처럼 for 문 없이 열(벡터) 전체에 계산을 한 번에 적용하는 것을 **벡터 연산**이라고 합니다. 넘파이와 판다스가 내부의 빠른 코드로 계산하므로 백만 줄도 for 문보다 수십~수백 배 빠릅니다. 열끼리 계산(`df["a"] * df["b"]`), 비교(`df["a"] > 0`), 요약(`.sum( )`, `.mean( )`)이 모두 벡터 연산이고, 참·거짓 열의 `.sum( )`은 참의 개수, `.mean( )`은 참의 비율입니다.

**❸ 기울기와 절편 (식 8-3)**

In [ ]:
sxy = study["편차곱"].sum()             # (식 8-3)의 분자
sxx = study["x편차²"].sum()             # (식 8-3)의 분모
w = sxy / sxx                            # 기울기
b = y.mean() - w * x.mean()              # 절편
print(f"기울기 w = {sxy} ÷ {sxx} = {w}")
print(f"절편 b = {y.mean()} - {w} × {x.mean()} = {b}")

손 계산과 같은 3.75와 53.5입니다. 절편 식은 직선이 평균점 (6, 76)을 지나도록 높이를 맞춘 것입니다.

**❹ 예측값, 잔차, R² (식 8-2, 8-4)**

In [ ]:
study["예측"] = b + w * x                # ŷ
study["잔차"] = y - study["예측"]         # y - ŷ
print(study[["공부시간", "점수", "예측", "잔차"]])
sse = (study["잔차"] ** 2).sum()
sst = (study["y편차"] ** 2).sum()
print(f"잔차의 합 {study['잔차'].sum()}, SSE {sse}, SST {sst}")
print(f"R² = 1 - {sse} ÷ {sst} = {1 - sse / sst:.3f}")

- `study["예측"] = b + w * x`: 다섯 학생의 예측값 ŷ을 한 번에 구합니다.
- `study["잔차"] = y - study["예측"]`: 잔차는 "실제 - 예측"입니다. 순서를 바꾸면 부호가 반대가 됩니다.
- `sse`, `sst`: 잔차의 제곱합과, 평균에서 잰 처음 흩어짐(총제곱합)입니다.

잔차의 합은 0입니다. 최소제곱 직선은 늘 그렇습니다. SSE 7.5는 처음 흩어짐 570의 1.3%뿐이라 R²는 0.987입니다.

**❺ 라이브러리로 검산하기**

손으로 짠 코드가 맞는지 이미 검증된 함수로 확인하는 습관을 들입니다.

In [ ]:
print("np.polyfit:", np.polyfit(x, y, 1))           # [기울기, 절편]
print("상관계수의 제곱:", round(x.corr(y) ** 2, 3))

`np.polyfit`도 [기울기, 절편] = [3.75, 53.5]를 내고, 상관계수의 제곱도 R²와 같은 0.987입니다.

**❻ 기울기를 바꿔 가며 SSE 그리기**

In [ ]:
ws = np.linspace(2.5, 5.0, 51)          # 2.5~5 사이 기울기 51개
def sse_of(v):                           # 기울기 v일 때의 SSE
    bb = y.mean() - v * x.mean()         # 직선이 평균점을 지나게
    return ((y - (bb + v * x)) ** 2).sum()
for v in [3.0, 3.5, 3.75, 4.0, 4.5]:
    print(f"w = {v:4} → SSE = {sse_of(v):5.1f}")
plt.figure(figsize=(5, 3.2))
plt.plot(ws, [sse_of(v) for v in ws], color="gray")
plt.scatter([w], [sse], color="#C8234B", zorder=3)
plt.xlabel("기울기 w")
plt.ylabel("잔차 제곱합(SSE)")
plt.show()

- `np.linspace(2.5, 5.0, 51)`: 2.5부터 5.0까지 고르게 떨어진 값 51개를 만듭니다.
- `def sse_of(v):`: 기울기 v를 받아 SSE를 돌려주는 함수를 만듭니다. 같은 계산을 여러 번 쓸 때 함수로 묶습니다.
- `[sse_of(v) for v in ws]`: 리스트 컴프리헨션입니다. ws의 값마다 함수를 불러 결과를 목록으로 모읍니다.
- `zorder=3`: 빨간 점을 선 위에 그리라는 뜻입니다.

**결과 읽기.** 기울기를 3.75에서 0.25 바꾸면 SSE가 10.0, 0.75 바꾸면 30.0으로 커집니다. 양쪽으로 똑같이 커지는 아래로 볼록한 그릇 모양이고 바닥이 하나뿐입니다. 그래서 바닥(미분해서 0이 되는 곳)을 공식으로 구할 수 있고, 그 공식이 (식 8-3)입니다. 코랩 실습 8-1의 경사하강법은 이 그릇의 바닥을 걸어서 찾아갑니다.

**[직접 해 보기]** [연습문제 8-1]의 점 네 개 (1, 2), (2, 3), (3, 5), (4, 6)으로 같은 계산을 하시오.

In [ ]:
# 여기에 직접 코드를 써 보십시오.

**정답 코드** (직접 해 본 뒤에 실행해 보십시오)

In [ ]:
pts = pd.DataFrame({"x": [1, 2, 3, 4], "y": [2, 3, 5, 6]})
dx, dy = pts["x"] - pts["x"].mean(), pts["y"] - pts["y"].mean()
w4 = (dx * dy).sum() / (dx ** 2).sum()
b4 = pts["y"].mean() - w4 * pts["x"].mean()
sse4 = ((pts["y"] - (b4 + w4 * pts["x"])) ** 2).sum()
r2_4 = 1 - sse4 / (dy ** 2).sum()
print(f"ŷ = {b4:.1f} + {w4:.1f}x, SSE {sse4:.1f}, R² {r2_4:.2f}")

> **[자주 하는 실수] 정수와 실수, 그리고 변수 이름 덮어쓰기**
>
> 이 실습은 기울기를 `w`, 절편을 `b`라 불렀습니다. 실습 예제 8-3에서는 `b`가 기울기였습니다. 같은 이름에 새 값을 넣으면 옛 값은 사라지니, 노트북 중간의 셀을 다시 실행할 때는 그 셀이 어떤 이름을 덮어쓰는지 확인합니다.
>
> 엑셀처럼 `x^2`로 쓰면 오류는 안 나지만 전혀 다른 계산(비트 연산)이 됩니다. 거듭제곱은 `x ** 2`입니다.

#### [코랩 실습 8-1] 경사하강법으로 같은 직선 찾기

> 📘 책에는 문제와 핵심 결과만 있고, 코드는 이 노트북에만 있습니다.

w = 0, b = 0에서 출발해 학습률 0.01로 3,000걸음 경사하강법을 하시오(04절 4.5, 06절 6.1).

**❶ 한 걸음만 손으로 따라가기**

In [ ]:
xa, ya = x.to_numpy(float), y.to_numpy(float)
def loss(w_, b_):                        # 손실 = 잔차 제곱의 평균
    return ((b_ + w_ * xa - ya) ** 2).mean()
err = (0 + 0 * xa) - ya                  # w = 0, b = 0일 때 예측 - 실제
gw, gb = 2 * (err * xa).mean(), 2 * err.mean()   # 손실의 기울기
print("처음 손실", loss(0, 0))
print("기울기 ∂L/∂w =", gw, " ∂L/∂b =", gb)
print("한 걸음 뒤 w =", 0 - 0.01 * gw, " b =", 0 - 0.01 * gb,
      " 손실", round(loss(-0.01 * gw, -0.01 * gb), 1))

- `x.to_numpy(float)`: 판다스 열을 넘파이 배열로 바꿉니다. 반복 계산이 조금 더 빠르고 단순합니다.
- `loss(w_, b_)`: 손실 L = 잔차 제곱의 평균입니다(SSE를 개수 5로 나눈 값).
- `2 * (err * xa).mean()`: 손실을 w로 미분한 ∂L/∂w, `2 * err.mean()`은 ∂L/∂b입니다.
- `0 - 0.01 * gw`: (식 8-6) w ← w - η ∂L/∂w를 한 번 적용한 것입니다.

기울기가 음수(-972)이니 w를 키우는 쪽이 내리막입니다. 한 걸음에 손실이 5,890에서 547.8로 줄었습니다.

**❷ 3,000걸음 반복하고 손실 그리기**

In [ ]:
w_gd, b_gd, lr = 0.0, 0.0, 0.01          # 출발점과 학습률
losses = []
for step in range(3000):
    err = (b_gd + w_gd * xa) - ya        # 예측 - 실제
    losses.append((err ** 2).mean())
    if step in (0, 100, 1000):
        print(f"{step:4d}걸음: w = {w_gd:6.3f}, b = {b_gd:6.3f},"
              f" 손실 = {losses[-1]:8.2f}")
    w_gd -= lr * 2 * (err * xa).mean()   # (식 8-6)
    b_gd -= lr * 2 * err.mean()
print(f"3000걸음: w = {w_gd:.3f}, b = {b_gd:.3f}, 손실 = {losses[-1]:.2f}")
plt.figure(figsize=(5, 3.2))
plt.plot(losses, color="#C8234B")
plt.yscale("log")                        # 손실을 로그 눈금으로
plt.xlabel("걸음 수")
plt.ylabel("손실(로그 눈금)")
plt.show()

- `for step in range(3000):`: 들여쓴 줄을 3,000번 되풀이합니다. `step`은 0부터 2999까지 바뀝니다.
- `losses.append(…)`: 걸음마다 손실을 목록 끝에 붙여 두었다가 그래프로 그립니다.
- `if step in (0, 100, 1000):`: 몇 걸음에서만 중간 상태를 찍습니다.
- `w_gd -= …`: `w_gd = w_gd - …`의 줄임입니다.
- `plt.yscale("log")`: 세로축을 로그 눈금으로 바꿔, 5,890에서 1.5까지 크게 변하는 손실을 한 그림에 담습니다.

**결과 읽기.** 3,000걸음 뒤 공식의 답(3.75, 53.5)에 도착했습니다. 100걸음에서 w가 8.73까지 지나쳤다가 돌아오는 것은 절편 b가 아직 작아 기울기가 그 몫을 대신 떠맡았기 때문입니다. 처음에 빨리 줄고 뒤로 갈수록 천천히 주는 모양은 신경망의 학습 곡선과 똑같습니다.

> **[개념 노트] 학습률과 하이퍼파라미터**
>
> **학습률**은 한 번에 계수를 얼마나 크게 고칠지 정하는 숫자입니다. 너무 작으면 바닥까지 오래 걸리고, 너무 크면 골짜기를 넘어 튕겨 나갑니다(0.05로 바꾸면 손실이 끝없이 커집니다). 이처럼 모델이 배우는 값(w, b)이 아니라 사람이 미리 정하는 값을 **하이퍼파라미터**라고 합니다. 학습률, 걸음 수, 릿지의 λ, 랜덤 포레스트의 나무 수가 그 예입니다.

**[직접 해 보기]** 학습률을 0.001, 0.01, 0.02, 0.05로 바꿔 3,000걸음 뒤의 w, b, 손실을 견주시오.

In [ ]:
# 여기에 직접 코드를 써 보십시오.

**정답 코드** (직접 해 본 뒤에 실행해 보십시오)

In [ ]:
for lr_ in [0.001, 0.01, 0.02, 0.05]:
    w_, b_ = 0.0, 0.0
    for step in range(3000):
        e = (b_ + w_ * xa) - ya
        w_ -= lr_ * 2 * (e * xa).mean()
        b_ -= lr_ * 2 * e.mean()
    L = loss(w_, b_)
    print(f"학습률 {lr_}:", f"w = {w_:.2f}, b = {b_:.2f}, 손실 {L:.2f}"
          if np.isfinite(L) else "손실이 끝없이 커짐(발산)")

0.001은 너무 느려 3,000걸음으로는 바닥에 못 닿고, 0.05는 골짜기를 건너뛰다 튕겨 나가 손실이 끝없이 커집니다(발산). 알맞은 학습률은 데이터의 크기(여기서는 x가 10까지)에 따라 달라집니다.

#### [실습 예제 8-5 ★] 서울 태양광과 날씨 기록 잇기

한국전력거래소 파일에서 서울 태양광을 고르고, 기상청 서울 관측 자료와 시각으로 이으시오. 일사 빈칸도 세시오(03절 3.1~3.2).

**❶ 발전량 파일 읽기**

In [ ]:
SOLAR = "kpx_지역별_시간별_태양광풍력_발전량_2025.csv"
kpx = pd.read_csv(DATA + SOLAR)
print(kpx.shape)
print(kpx.head(3))

- `DATA + SOLAR`: 폴더 이름과 파일 이름을 이어 경로를 만듭니다. 파일 이름을 변수에 담아 두면 해가 바뀐 파일로 갈아 끼우기 쉽습니다.

166,440줄, 다섯 열입니다. 한 줄은 "어느 날, 몇 시, 어느 지역, 어느 연료원이 판 전기의 양"입니다. 지역과 연료원이 섞여 있어 이대로는 서울만 볼 수 없습니다.

**❷ 지역과 연료원 값 보기**

In [ ]:
print(kpx["지역"].unique().tolist()[:5], "…", kpx["지역"].nunique(), "곳")
print(kpx["연료원"].value_counts())

- `unique()`: 열에 나오는 값을 겹치지 않게 모읍니다. `nunique()`는 그 개수입니다.
- `.tolist()[:5]`: 목록으로 바꿔 앞의 다섯 개만 봅니다.

18개 지역이 있고, 서울은 "서울"이 아니라 "서울시"로 적혀 있습니다. 값을 확인하지 않고 `== "서울"`로 고르면 오류 없이 빈 표가 나오니, 고르기 전에 반드시 값을 봅니다. 태양광 148,920줄은 17개 지역 × 8,760시각이고, 풍력은 2개 지역뿐입니다.

**❸ 서울 태양광만 고르기**

In [ ]:
seoul = kpx[(kpx["지역"] == "서울시")
            & (kpx["연료원"] == "태양광")].copy()   # 조건 두 개를 & 로
print(seoul.shape)
print(seoul["전력거래량(MWh)"].describe().round(2))

- `(조건1) & (조건2)`: 두 조건이 모두 참인 줄만 고릅니다. 조건마다 괄호를 씌웁니다.
- 줄이 길어 괄호 안에서 줄을 바꿨습니다. 괄호가 열려 있으면 파이썬은 다음 줄을 이어 읽습니다.

8,760줄은 365일 × 24시간이니 빠진 시각이 없습니다. 가운데 값(50%)이 0.16MWh로 작은 것은 밤이 절반이기 때문이고, 가장 많은 시각은 14.52MWh입니다.

**❹ "끝나는 시각" 만들기**

In [ ]:
seoul["일시"] = (pd.to_datetime(seoul["거래일"])
                 + pd.to_timedelta(seoul["거래시간"], unit="h"))
seoul = seoul.rename(columns={"전력거래량(MWh)": "발전량"})
seoul = seoul[["일시", "발전량"]]
print(seoul.iloc[11:14])                # 1월 1일 12~14시
print(f"한 해 {seoul['발전량'].sum():,.0f}MWh")

- `pd.to_datetime(seoul["거래일"])`: "2025-01-01" 같은 글자를 날짜로 바꿉니다.
- `pd.to_timedelta(seoul["거래시간"], unit="h")`: 13을 "13시간"이라는 시간 길이로 바꿉니다. 날짜에 더하면 "2025-01-01 13:00"이 됩니다.
- `rename(columns={옛 이름: 새 이름})`: 긴 열 이름을 짧게 바꿉니다.
- `seoul[["일시", "발전량"]]`: 쓸 두 열만 남깁니다.
- `iloc[11:14]`: 위치로 12~14번째 줄을 봅니다(0부터 세므로 11이 12번째).

장부의 "13시"는 12시부터 13시까지의 한 시간이므로, 끝나는 시각 13:00으로 맞췄습니다. 기상청의 13시 일사도 같은 한 시간에 쌓인 양이라 두 기록이 이 시각으로 맞물립니다. 왼쪽 숫자 203~205는 원래 표의 줄 번호가 따라온 것입니다.

> **[개념 노트] 날짜·시간 열 다루기**
>
> 글자 "2025-01-01 13:00"을 `pd.to_datetime( )`으로 바꾸면 `.dt.hour`(시), `.dt.month`(월), `.dt.dayofweek`(요일, 월요일 0)를 꺼내고 날짜끼리 비교하거나 더할 수 있습니다. 서로 다른 기관의 기록을 이을 때는 시각이 "시작"인지 "끝"인지(이 장의 13시 문제), 시간대가 한국 표준시인지 UTC인지, 하루의 끝이 "24시"인지 "다음 날 0시"인지를 확인합니다. 이것이 어긋나면 오류 없이 숫자만 조용히 틀립니다.

**❺ 날씨 파일 읽기**

In [ ]:
ASOS = "asos_서울108_시간자료_전체열_2023_2026.csv"
names = {"일시": "일시", "기온(°C)": "기온", "습도(%)": "습도",
         "일사(MJ/m2)": "일사", "전운량(10분위)": "운량", "적설(cm)": "적설"}
wx = pd.read_csv(DATA + ASOS, usecols=list(names)).rename(columns=names)
wx["일시"] = pd.to_datetime(wx["일시"])
print(wx.shape, wx["일시"].min(), "~", wx["일시"].max())
print(wx[wx["일시"] == "2025-01-01 13:00"])

- `names`: 원래 열 이름(왼쪽)과 짧은 이름(오른쪽)을 짝지은 사전입니다.
- `usecols=list(names)`: 사전의 키, 곧 필요한 여섯 열만 읽습니다. 열이 많은 파일을 빨리, 가볍게 읽습니다.
- `wx[wx["일시"] == "2025-01-01 13:00"]`: 날짜 열은 글자와 바로 견줄 수 있습니다.

기상청 파일은 2023년부터 2026년 10월까지 33,241시각이라 2025년보다 깁니다. 적설의 `NaN`은 빈칸, 곧 눈이 쌓이지 않았다는 뜻입니다.

**❻ 같은 시각끼리 옆으로 붙이기**

In [ ]:
df = seoul.merge(wx, on="일시", how="left")   # 발전량 표에 날씨를 붙인다
print(df.shape)
print(df.iloc[11:14])

- `merge(wx, on="일시", how="left")`: 두 표에서 "일시"가 같은 줄을 옆으로 붙입니다. `how="left"`는 왼쪽(발전량) 표의 줄을 모두 남기고, 짝이 없으면 빈칸으로 둡니다.

줄 수는 그대로 8,760이고, 열은 2 + 5 = 7개가 됐습니다. 1월 1일 13시에 일사 1.73, 기온 7.8℃가 붙었습니다. 붙인 뒤에는 줄 수가 그대로인지 늘 확인합니다. 늘었다면 오른쪽 표에 같은 시각이 두 번 있다는 뜻입니다.

> **[개념 노트] 두 표를 잇는 네 가지 방법: merge**
>
> `A.merge(B, on="열", how=…)`는 열 값이 같은 줄끼리 옆으로 붙입니다. `"inner"`는 양쪽에 다 있는 줄만, `"left"`는 왼쪽 표의 줄을 모두(짝이 없으면 빈칸), `"right"`는 오른쪽 표의 줄을 모두, `"outer"`는 어느 한쪽에라도 있는 줄을 모두 남깁니다. 엑셀의 VLOOKUP, SQL의 JOIN과 같은 일입니다. 잇고 나면 줄 수(여기서는 8,760)와 새로 생긴 빈칸 수를 꼭 셉니다. 한쪽에 같은 열 값이 두 번 있으면 줄이 조용히 불어나기 때문입니다. `validate="one_to_one"`을 넣으면 이런 겹침에서 판다스가 오류를 내 줍니다.

**❼ 시각이 맞물렸는지 확인하기**

In [ ]:
for h in [-1, 0, 1]:                     # 날씨 시각을 한 칸씩 밀어 본다
    moved = wx[["일시", "일사"]].copy()
    moved["일시"] = moved["일시"] + pd.Timedelta(hours=h)
    m = seoul.merge(moved, on="일시")
    r_ = m["발전량"].corr(m["일사"].fillna(0))
    print(f"날씨 시각을 {h:+d}시간 밀면 상관계수 {r_:.3f}")

- `pd.Timedelta(hours=h)`: h시간이라는 길이입니다. 날씨의 시각을 -1, 0, +1시간 밀어 붙여 봅니다.
- `fillna(0)`: 밤의 빈 일사를 0으로 채운 뒤 상관계수를 잽니다.

밀지 않았을 때 상관계수가 0.971로 가장 높습니다. 한 시간 밀면 0.899, 0.921로 떨어지니 두 기록의 시각이 제대로 맞물렸습니다.

**❽ 빈칸 세기**

In [ ]:
print(df.isna().sum())                   # 열마다 빈칸 수
df["시각"] = df["일시"].dt.hour
blank = df["일사"].isna()
noon = blank & df["시각"].between(8, 18)
print("일사 빈칸", blank.sum(), "/ 한낮", noon.sum(),
      df.loc[noon, "일시"].dt.date.astype(str).unique().tolist())

- `df.isna().sum()`: 열마다 빈칸(`NaN`)의 개수입니다. 참·거짓을 더하면 참의 개수가 됩니다.
- `dt.hour`: 날짜·시각 열에서 시(0~23)만 꺼냅니다.
- `between(8, 18)`: 8 이상 18 이하인지 묻습니다.
- `blank & …`: 일사가 비었으면서 한낮인 줄입니다.

일사 빈칸 3,978시각 가운데 한낮은 9시각, 모두 9월 13일입니다. 적설 빈칸 8,367은 눈이 없던 시각이라 문제가 아닙니다. 같은 빈칸이라도 열마다, 시각마다 뜻이 다릅니다.

**❾ 밤의 발전량과 시각별 평균 그리기**

In [ ]:
night = blank & ~noon                    # ~ 는 '아니다'
print("밤 빈칸 가운데 발전량 > 0:", (df.loc[night, "발전량"] > 0).sum(),
      "시각, 합", round(df.loc[night, "발전량"].sum(), 1), "MWh")
df.groupby("시각")["발전량"].mean().plot.bar(color="gray", figsize=(6, 3))
plt.ylabel("평균 발전량(MWh)")
plt.xlabel("시각(그 시각까지 한 시간)")
plt.xticks(range(0, 24, 3), range(0, 24, 3), rotation=0)
plt.show()

- `~noon`: `~`는 "아니다"입니다. 빈칸이면서 한낮이 아닌 줄, 곧 밤입니다.
- `groupby("시각")["발전량"].mean()`: 시각(0~23)마다 묶어 평균을 냅니다. 이어 붙인 `.plot.bar()`가 곧바로 막대그래프를 그립니다.
- `plt.xticks(range(0, 24, 3), …, rotation=0)`: 가로축 눈금을 3시간 간격으로, 글자를 눕히지 않고 씁니다.

**결과 읽기.** 밤 빈칸 1,359시각에 발전량이 조금씩(합 137.2MWh, 한 해의 1% 미만) 거래되었습니다. 그래프에서도 19~23시에 작은 막대가 보입니다. 햇빛으로 설명할 수 없는 값이므로, 다음 실습부터는 일사가 0보다 큰 시각만 씁니다(03절 3.2). 한낮의 빈칸 9시각은 "측정 못 함"이라 함께 빠집니다.

> **[개념 노트] 결측값: 지울까, 채울까, 표시할까**
>
> 빈칸(결측값, `NaN`)을 만나면 먼저 **왜 비었는지**를 묻습니다. "밤이라 0"인 빈칸은 0으로 채우고(`fillna(0)`), "측정기가 멈춰서 모름"인 빈칸은 지우거나(`dropna( )`), 빈칸 자체가 정보일 수 있으면(설문에서 소득을 밝히지 않음) "빈칸이었음" 열을 따로 만들어 표시합니다(`isna( )`). 이유를 묻지 않고 한꺼번에 평균으로 채우면, 밤의 일사가 한낮 평균으로 바뀌는 엉뚱한 데이터가 됩니다. 몇 줄을 어떻게 처리했는지는 보고서에 적어 둡니다.

**[직접 해 보기]** 달마다 발전량의 합을 구하시오. 가장 많이 만든 달과 가장 적게 만든 달은?

In [ ]:
# 여기에 직접 코드를 써 보십시오.

**정답 코드** (직접 해 본 뒤에 실행해 보십시오)

In [ ]:
month_sum = df.groupby(df["일시"].dt.month)["발전량"].sum()
print(month_sum.round(0).astype(int).to_dict())

4월(2,397MWh)이 가장 많고 12월(1,215MWh)이 가장 적습니다. 해가 가장 긴 6월이 아니라 4월이 1등인 것은, 여름에는 장마와 더위가 발전을 깎기 때문입니다. 실습 예제 8-7에서 이 까닭을 변수로 만듭니다.

> **[자주 하는 실수] 글자가 깨지거나, 표가 비거나, 줄이 늘거나**
>
> 공공데이터포털에서 받은 원본 CSV를 그냥 읽으면 `UnicodeDecodeError`가 나거나 한글이 "¼­¿ï"처럼 깨집니다. 한글 윈도 인코딩이기 때문이니 `pd.read_csv(파일, encoding="cp949")`로 읽습니다. 수업 자료의 정리본은 UTF-8이라 그대로 읽힙니다.
>
> 조건으로 고른 표가 비었으면(`shape`가 (0, 5)) 값의 철자를 의심합니다. "서울시"와 "서울", "태양광"과 "태양광 " 같은 차이입니다.
>
> 두 표의 "일시"가 하나는 날짜, 하나는 글자이면 `merge`가 짝을 하나도 찾지 못해 날씨 열이 모두 빈칸이 됩니다. 붙이기 전에 두 열 모두 `pd.to_datetime( )`으로 바꿉니다.

> **[더 알아보기] 원자료를 직접 받아 정리하기**
>
> 수업 자료의 세 파일은 공공데이터포털과 기상자료개방포털에서 받은 원자료를 정리한 것입니다. 원자료는 한글 윈도 인코딩(cp949)이라 `encoding="cp949"`로 읽어야 글자가 깨지지 않습니다. 기상청 자료는 한 번에 받을 수 있는 기간이 정해져 있어 여러 파일로 나뉘어 옵니다. 그래서 `pd.concat( )`으로 이어 붙이고, 겹친 시각을 `drop_duplicates("일시")`로 지웁니다. 일사와 강수량의 빈칸은 지우지 않습니다. 밤이거나 비가 안 온 시각이라는 뜻이 있기 때문입니다(3.2). 노트북의 "원자료 정리" 셀에 전체 코드가 있습니다.

#### [실습 예제 8-6 ★] 일사 하나로 회귀하고 기준선과 겨루기

해가 있던 시각으로 "발전량 ~ 일사" 회귀를 하고, 1~10월로 배우고 11~12월로 시험해 기준선과 견주시오(03절 3.3).

**❶ 해가 있던 시각으로 회귀하기**

In [ ]:
import statsmodels.formula.api as smf
day = df[df["일사"] > 0].copy()          # 해가 있던 시각만
print(len(day), "시각")
m1 = smf.ols("발전량 ~ 일사", data=day).fit()   # y ~ x 꼴의 식
print(m1.params.round(3))
print("R²", round(m1.rsquared, 3))

- `df[df["일사"] > 0]`: 일사가 0보다 큰 줄만 남깁니다. 빈칸(`NaN`)은 어떤 비교에도 거짓이라 함께 빠집니다.
- `smf.ols("발전량 ~ 일사", data=day)`: "발전량을 일사로 설명하라"는 식을 글자 그대로 씁니다. ols는 최소제곱법(Ordinary Least Squares)의 머리글자입니다.
- `.fit()`: 최소제곱으로 계수를 구합니다. 결과(`m1`)에 계수(`.params`), R²(`.rsquared`), 잔차(`.resid`)가 들어 있습니다.

"발전량 = 0.255 + 3.966 × 일사"이고 R²는 0.910입니다. 일사가 1 늘면 서울의 태양광이 한 시간에 약 4MWh를 더 만듭니다.

> **[코드 읽기] 식으로 쓰는 회귀 `"발전량 ~ 일사"`**
>
> `~` 왼쪽이 종속변수, 오른쪽이 독립변수입니다. 독립변수를 더하려면 `"발전량 ~ 일사 + 기온"`처럼 `+`로 잇고, 곱한 효과는 `일사:기온`, 제곱은 `I(기온 ** 2)`, 범주는 `C(시각)`으로 씁니다. 절편은 저절로 들어가며, 빼려면 `"발전량 ~ 일사 - 1"`로 씁니다. 열 이름이 한글이어도 되지만 빈칸이나 괄호가 있으면 `Q("전력거래량(MWh)")`처럼 감싸야 하니, 미리 짧은 이름으로 바꿔 두는 편이 편합니다.

**❷ 계수 표 읽기**

In [ ]:
print(m1.summary().tables[1])            # 계수 표만

일사의 계수 3.9665, 표준오차 0.018, 95% 구간 [3.930, 4.003]입니다. 구간이 아주 좁은 것은 데이터가 4,616시각이나 되기 때문입니다.

> **[개념 노트] 회귀 결과표 읽는 법**
>
> **coef**는 계수(일사 1이 늘 때 발전량이 3.97MWh 는다), **std err**(표준오차)는 데이터를 다시 모으면 계수가 흔들릴 크기, **t**는 계수 ÷ 표준오차입니다. **P>|t|**(p값)는 "이 변수가 사실은 효과가 없다면 이만큼 또는 더 큰 t가 나올 확률"로, 보통 0.05보다 작으면 효과가 있다고 봅니다. **[0.025 0.975]**는 계수의 95% 신뢰구간입니다.
>
> 데이터가 수천 줄이면 아주 작은 효과도 p값이 0.000이 되므로 p값보다 계수의 크기와 단위를 먼저 읽으세요. 다른 변수를 더하면 계수가 바뀝니다(02절 중고차). 또 p값과 구간은 잔차가 서로 독립이라는 가정 위에서 계산하므로, 더빈-왓슨이 2에서 멀면(M3는 0.88) 실제보다 좁게 나옵니다.

**❸ 밤까지 넣으면?**

In [ ]:
night_in = df.fillna({"일사": 0})        # 빈칸을 0으로 채워 밤까지 넣으면
m_all = smf.ols("발전량 ~ 일사", data=night_in).fit()
print("밤까지 넣은 R²", round(m_all.rsquared, 3))

- `df.fillna({"일사": 0})`: 일사 열의 빈칸만 0으로 채운 새 표를 만듭니다. 원래 `df`는 바뀌지 않습니다.

R²가 0.910에서 0.942로 오릅니다. 하지만 "밤에는 0"이라는 쉬운 문제를 섞은 덕분이지 모델이 나아진 것이 아닙니다. 그래서 이 장은 해가 있던 시각만 씁니다.

**❹ 학습과 시험으로 나누기**

In [ ]:
CUT = "2025-11-01"
train = day[day["일시"] < CUT]           # 1~10월로 배우고
test = day[day["일시"] >= CUT]           # 11~12월로 시험
print("학습", len(train), "시각 / 시험", len(test), "시각")
print(test["일시"].min(), "~", test["일시"].max())

- `day["일시"] < CUT`: 날짜 열과 "2025-11-01" 같은 글자를 바로 견줄 수 있습니다.
- 무작위로 섞지 않고 날짜로 자르는 것이 핵심입니다. 내일을 맞힐 모델은 아직 오지 않은 날로 시험합니다.

1~10월 3,968시각으로 배우고, 11월 1일 8시부터 12월 31일 18시까지 648시각으로 시험합니다.

**❺ 채점 함수와 기준선 셋**

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score
def score(name, y_true, y_pred):         # 같은 채점표로 견준다
    mae = mean_absolute_error(y_true, y_pred)
    print(f"{name:8s} MAE {mae:5.2f}MWh  R² {r2_score(y_true, y_pred):6.3f}")
score("평균 찍기", test["발전량"], np.full(len(test), train["발전량"].mean()))
by_hour = train.groupby("시각")["발전량"].mean()       # 시각별 평균표
score("시각별 평균", test["발전량"], test["시각"].map(by_hour))
yday = df.set_index("일시")["발전량"].shift(24)        # 24시각 전 값
score("어제 같은 시", test["발전량"], test["일시"].map(yday))

- `def score(name, y_true, y_pred):`: 이름, 실제 값, 예측값을 받아 MAE와 R²를 한 줄로 찍는 함수입니다. 모델을 바꿔도 잣대는 그대로라서 공정한 비교가 됩니다.
- `f"{name:8s}"`: 이름을 8칸에 맞춰 써서 줄이 가지런해지게 합니다.
- `np.full(len(test), 값)`: 같은 값(학습 기간 평균)을 시험 시각 수만큼 늘어놓습니다. "평균으로 찍기"입니다.
- `train.groupby("시각")["발전량"].mean()`: 학습 기간의 시각별 평균표입니다. `test["시각"].map(by_hour)`가 시험 시각마다 그 평균을 찾아옵니다.
- `shift(24)`: 줄을 24칸 아래로 밀어, 각 시각 옆에 24시간 전 값을 놓습니다. "어제 같은 시각"입니다.

가장 단순한 예측들의 성적입니다. "평균으로 찍기"의 R²가 음수인 것은 시험 기간 자신의 평균으로 찍는 것보다도 못하다는 뜻입니다([연습문제 8-4]). 가장 나은 기준선은 어제 같은 시각(1.94MWh)입니다.

**❻ 일사 회귀를 같은 잣대로 채점하기**

In [ ]:
fit = smf.ols("발전량 ~ 일사", data=train).fit()       # 학습 데이터로만
score("일사 회귀", test["발전량"], fit.predict(test))
res_t = test["발전량"] - fit.predict(test)
print(f"시험 잔차의 평균 {res_t.mean():+.2f}MWh")

- `smf.ols(…, data=train)`: 학습 기간으로만 배웁니다. 시험 기간이 섞여 들어가면 데이터 누수입니다.
- `fit.predict(test)`: 시험 표의 일사로 예측합니다. 열 이름만 같으면 됩니다.

일사 회귀의 오차 0.98MWh는 가장 나은 기준선(1.94)의 절반입니다. [표 8-1]의 네 줄이 모두 나왔습니다. 시험 잔차의 평균이 +0.59로 0이 아닌 것은 11~12월에 예측이 실제보다 낮게 치우쳤다는 뜻이고, 실습 예제 8-7이 그 까닭을 찾습니다.

**❼ 랜덤 분할과 견주고, 산점도 그리기**

In [ ]:
from sklearn.model_selection import train_test_split
r_tr, r_te = train_test_split(day, test_size=0.2, random_state=42)
fit_r = smf.ols("발전량 ~ 일사", data=r_tr).fit()
score("랜덤 분할", r_te["발전량"], fit_r.predict(r_te))
plt.figure(figsize=(5, 4))
plt.scatter(day["일사"], day["발전량"], s=4, alpha=0.3, color="gray")
xs = np.array([0, 3.9])
plt.plot(xs, m1.params["Intercept"] + m1.params["일사"] * xs, color="#C8234B")
plt.xlabel("일사(MJ/m², 한 시간)")
plt.ylabel("서울 태양광 발전량(MWh)")
plt.show()

- `train_test_split(day, test_size=0.2, random_state=42)`: 줄을 무작위로 섞어 80%는 학습, 20%는 시험으로 나눕니다. `random_state`를 정해 두면 누가 돌려도 같은 결과가 나옵니다.

**결과 읽기.** 랜덤 분할의 R² 0.920은 시간 분할(0.875)보다 좋아 보이지만, 같은 날의 이웃 시각을 미리 보고 본 시험이라 부풀려진 점수입니다(데이터 누수, 03절 3.3). 산점도는 직선을 따라 모여 있지만 일사가 클수록 위아래로 넓게 퍼집니다. 같은 햇빛에서도 발전량이 다른 까닭이 따로 있다는 신호입니다.

> **[개념 노트] 스태츠모델과 사이킷런, 언제 무엇을 쓸까**
>
> **스태츠모델**은 통계학의 도구로, 식을 글자로 쓰고 계수의 표준오차·p값·구간을 보고서로 보여 줍니다. "무엇이 얼마나 영향을 주는가"를 설명할 때 씁니다. **사이킷런**은 머신러닝의 도구로, 모든 모델이 `fit(X, y)`와 `predict(X)`라는 같은 사용법을 가져서 "무엇이 가장 잘 맞히는가"를 견줄 때 씁니다. 이 장은 모델을 스태츠모델로 만들고, 채점과 여러 모델 비교(실습 예제 8-10)는 사이킷런으로 합니다.

**[직접 해 보기]** 1~6월로 배우고 7~12월로 시험하면 일사 회귀의 성적은 어떻게 바뀌나요?

In [ ]:
# 여기에 직접 코드를 써 보십시오.

**정답 코드** (직접 해 본 뒤에 실행해 보십시오)

In [ ]:
cut2 = "2025-07-01"
tr2, te2 = day[day["일시"] < cut2], day[day["일시"] >= cut2]
fit2 = smf.ols("발전량 ~ 일사", data=tr2).fit()
score("7~12월 시험", te2["발전량"], fit2.predict(te2))

시험 MAE가 0.87MWh입니다. 시험 기간을 바꾸면 성적도 바뀌니, 성적을 말할 때는 어느 기간으로 시험했는지 함께 적습니다.

> **[자주 하는 실수] 학습 데이터로 채점하기**
>
> `m1`은 4,616시각 전체로 배운 모델입니다. 이것으로 시험 기간을 채점하면 답을 보고 푼 시험이 되어 성적이 좋게 나옵니다. 채점에 쓸 모델은 반드시 `train`으로만 다시 배웁니다(⑥의 `fit`).
>
> 시각별 평균표(`by_hour`)도 `train`으로만 만듭니다. `day` 전체로 만들면 시험 기간의 정보가 기준선에 새어 듭니다. 숫자 하나라도 시험 기간에서 계산한 값이 학습에 들어가면 누수입니다.

#### [실습 예제 8-7 ★] 겨울의 기울기와 온도 손실 변수

달마다 기울기를 견주고, 온도 손실 변수를 만들어 네 모델(M1~M4)을 견주시오(03절 3.4).

**❶ 달마다 직선을 따로 긋기**

In [ ]:
day["월"] = day["일시"].dt.month
slope = {}
for mo, g in day.groupby("월"):          # 달마다 따로 직선을 긋는다
    slope[mo] = smf.ols("발전량 ~ 일사", data=g).fit().params["일사"]
print(pd.Series(slope).round(2).to_dict())

- `day["일시"].dt.month`: 날짜에서 달(1~12)만 꺼내 새 열로 둡니다.
- `for mo, g in day.groupby("월"):`: 달마다 묶은 표를 (달, 그 달의 표) 쌍으로 하나씩 꺼냅니다.
- `.params["일사"]`: 회귀 결과에서 일사의 계수(기울기)만 꺼내 사전 `slope`에 담습니다.

이렇게 "묶고 → 묶음마다 계산하고 → 모으는" 흐름은 데이터 분석에서 가장 자주 쓰는 틀입니다(split-apply-combine).

**❷ 기울기를 막대그래프로**

In [ ]:
plt.figure(figsize=(6, 3))
plt.bar(list(slope), list(slope.values()), color="gray")
plt.xticks(range(1, 13))
plt.xlabel("월")
plt.ylabel("일사 1당 발전량(MWh)")
plt.show()

- `list(slope)`: 사전의 키(달 1~12), `list(slope.values())`: 값(기울기)입니다. 막대의 위치와 높이가 됩니다.
- `plt.xticks(range(1, 13))`: 가로축 눈금을 1~12에 하나씩 둡니다.

같은 일사 1에서 12월에는 5.33MWh, 7월에는 3.35MWh가 나옵니다. 햇빛이 같아도 여름에 덜 나오니, 계절에 따라 변하는 무언가가 발전량을 깎고 있습니다. 태양광 패널은 뜨거울수록 효율이 떨어진다는 지식이 그 답입니다.

**❸ 온도 손실 열 만들기**

In [ ]:
day["온도손실"] = day["일사"] * (day["기온"] - 25)
print(day[["일시", "일사", "기온", "온도손실"]].iloc[[2000, 2001, 4500]])
train, test = day[day["일시"] < CUT], day[day["일시"] >= CUT]

- `day["일사"] * (day["기온"] - 25)`: "햇빛이 셀 때 25℃보다 더운 만큼"입니다. 햇빛이 없으면 0, 25℃보다 추우면 음수가 됩니다.
- `iloc[[2000, 2001, 4500]]`: 위치로 세 줄을 골라 값이 뜻대로 계산됐는지 눈으로 확인합니다. 새 열을 만들면 늘 몇 줄을 찍어 봅니다.
- 열을 더했으니 `train`, `test`도 다시 나눕니다. 옛 `train`에는 새 열이 없습니다.

6월 8일 14시(일사 2.78, 30.1℃)는 온도 손실이 14.2로 크고, 12월 20일(10.2℃)은 음수입니다.

> **[개념 노트] 특성 공학: 지식을 변수로 바꾸기**
>
> 모델에 넣을 변수(머신러닝에서는 **특성**)를 원자료에서 새로 만드는 일을 **특성 공학**이라고 합니다. 온도 손실처럼 두 변수를 곱한 **상호작용**(한 변수의 효과가 다른 변수에 따라 달라짐), 기온 제곱 같은 **제곱항**(휘어진 관계, 미니 프로젝트), 날짜에서 꺼낸 요일·공휴일, 비율(매출 ÷ 방문자), 로그(집값처럼 오른쪽으로 긴 분포)가 대표적입니다. 좋은 특성은 "패널은 뜨거우면 효율이 떨어진다", "주말에는 공장이 쉰다" 같은 분야 지식을 열 하나로 옮긴 것입니다. 단, 시험 기간의 정보를 써서 특성을 만들면 누수가 됩니다.

**❹ 네 모델 견주기**

In [ ]:
forms = {"M1": "발전량 ~ 일사",
         "M2": "발전량 ~ 일사 + 기온 + 운량 + 습도",
         "M3": "발전량 ~ 일사 + 기온 + 운량 + 습도 + 온도손실",
         "M4": "발전량 ~ 일사 + 기온 + 운량 + 습도 + 온도손실 + C(시각)"}
for name, f in forms.items():
    fit = smf.ols(f, data=train).fit()
    tr = mean_absolute_error(train["발전량"], fit.fittedvalues)
    print(f"{name} 학습 MAE {tr:.2f} | ", end="")
    score("시험", test["발전량"], fit.predict(test))

- `forms`: 모델 이름과 식을 짝지은 사전입니다. 모델을 하나 더하려면 사전에 한 줄만 보태면 됩니다.
- `fit.fittedvalues`: 학습 데이터에 대한 예측값입니다. 학습 MAE와 시험 MAE를 나란히 봐야 과적합을 알아챕니다.
- `print(…, end="")`: 줄을 바꾸지 않고 이어서 다음 `score`의 출력을 붙입니다.

기온·운량·습도를 그냥 더한 M2는 시험 MAE가 0.98에서 0.96으로 거의 그대로인데, 온도 손실 열 하나를 더한 M3는 0.67로 30% 줄었습니다. 변수의 개수보다 뜻이 중요합니다. 학습 MAE와 시험 MAE가 비슷하니 과적합도 없습니다.

**❺ M3의 계수 읽기**

In [ ]:
m3 = smf.ols(forms["M3"], data=train).fit()
print(m3.params.round(4))
print(f"온도손실 p값 {m3.pvalues['온도손실']:.1e}")
drop = -m3.params["온도손실"] / m3.params["일사"]
print(f"기온 1℃마다 일사 1당 발전량 {drop:.1%} 감소")

온도 손실의 계수 -0.0491은 p값이 0에 가까워 우연으로 보기 어렵습니다. 이 계수를 일사 계수 3.6174로 나누면 "기온 1℃마다 같은 햇빛에서 만드는 전기가 1.4%씩 준다"가 됩니다. 패널 사양서의 온도 계수(보통 -0.3~-0.5%/℃)보다 크니, 겨울의 낮은 해 같은 계절 효과도 섞여 있습니다.

**❻ M4의 시각 더미 보기**

In [ ]:
m4 = smf.ols(forms["M4"], data=train).fit()
print("M4 계수 수", len(m4.params), "개")
print([k for k in m4.params.index if k.startswith("C(")][:4], "…")

> **[코드 읽기] `C(시각)`과 더미 변수**
>
> 시각은 숫자이지만, 9시와 10시의 차이가 15시와 16시의 차이와 같다고 볼 까닭이 없습니다. `C(시각)`은 "시각을 숫자가 아니라 범주로 다루라"는 뜻입니다. 스태츠모델은 시각마다 0과 1로 된 열을 만들어, 시각마다 따로 보정값을 둡니다. 해가 있던 시각이 6~20시의 15개라 첫 시각(6시)을 기준으로 빼고 14개 열이 생겼고, M3의 계수 6개와 합쳐 20개가 됐습니다. `C(시각)[T.7]`은 "6시보다 7시가 얼마나 더 많은가"로 읽습니다.

> **[개념 노트] 범주형 변수를 숫자로: 원-핫 인코딩**
>
> 지역, 요일처럼 숫자가 아닌 변수에 "서울 = 1, 부산 = 2, 대구 = 3"처럼 번호를 붙이면 "대구는 서울의 세 배"라는 엉뚱한 크기가 생깁니다. 그래서 범주마다 "이 범주인가?"를 묻는 0·1 열을 만듭니다. 이것이 **원-핫 인코딩**이고, 각 열이 **더미 변수**입니다. 회귀에서는 범주가 k개면 하나를 빼고 k − 1개 열을 만듭니다. 뺀 범주가 **기준 범주**가 되어 다른 계수는 "기준보다 얼마나 많은가"로 읽습니다. 다 넣으면 열의 합이 늘 1이라 절편과 겹치기 때문입니다(더미 변수 함정). 판다스는 `pd.get_dummies(df["지역"], drop_first=True)`, 스태츠모델은 `C(지역)`이 이 일을 합니다.

**[직접 해 보기]** 1~6월로 배우고 7~12월로 시험해도 온도 손실이 도움이 될까요?

In [ ]:
# 여기에 직접 코드를 써 보십시오.

**정답 코드** (직접 해 본 뒤에 실행해 보십시오)

In [ ]:
cut2 = "2025-07-01"
tr2, te2 = day[day["일시"] < cut2], day[day["일시"] >= cut2]
for name in ["M1", "M3"]:
    fit2 = smf.ols(forms[name], data=tr2).fit()
    score(name, te2["발전량"], fit2.predict(te2))

M1 0.87 → M3 0.76MWh로 여기서도 줄지만, 11~12월로 시험할 때(0.98 → 0.67)보다 차이가 작습니다([연습문제 8-5]).

> **[자주 하는 실수] 새 열을 만든 뒤 나누기를 잊기**
>
> ③에서 `day`에 온도손실 열을 만들기 전에 나눈 `train`, `test`에는 그 열이 없습니다. 그대로 M3를 돌리면 `PatsyError: … 온도손실`처럼 "그런 이름이 없다"는 오류가 납니다. 새 열을 만들면 학습·시험 표를 다시 나누거나, 처음부터 열을 다 만든 뒤에 나눕니다.

#### [실습 예제 8-8 ★] 예측구간과 크게 틀린 시각

M3로 한 조건의 예측값과 두 구간을 구하고, 시험 기간에 예측구간이 몇 %를 담는지 세시오. 가장 크게 틀린 시각을 찾고 잔차의 모양을 그리시오(03절 3.5).

**❶ 예측할 조건 표 만들기**

In [ ]:
cond = pd.DataFrame({"일사": [2.0], "기온": [10.0], "운량": [2], "습도": [50]})
cond["온도손실"] = cond["일사"] * (cond["기온"] - 25)
print(cond)
print("예측", m3.predict(cond).round(2).tolist())

- 모델에 넣을 새 데이터도 학습 때와 같은 열 이름을 가진 표여야 합니다. 값 하나라도 `[2.0]`처럼 리스트로 넣어 한 줄짜리 표를 만듭니다.
- 온도 손실 열도 학습 때와 똑같은 식으로 만듭니다. 학습과 예측에서 특성을 만드는 방법이 다르면 모델은 오류 없이 엉뚱한 값을 냅니다.

**❷ 신뢰구간과 예측구간**

In [ ]:
pf = m3.get_prediction(cond).summary_frame(alpha=0.05)
print(pf.round(2).T)                     # 세로로 돌려 보기

- `get_prediction(cond)`: 예측값과 함께 구간 계산에 필요한 정보를 돌려줍니다.
- `summary_frame(alpha=0.05)`: 95% 구간을 표로 만듭니다. 99%를 원하면 `alpha=0.01`입니다.
- `.T`: 표를 옆으로 눕혀(전치) 열 이름을 세로로 봅니다.

`mean`은 예측값 8.80, `mean_ci`는 평균의 신뢰구간 8.74~8.86, `obs_ci`는 한 시각의 예측구간 6.76~10.84입니다(obs는 관측값). 예측구간이 30배쯤 넓습니다.

**❸ 예측구간이 약속을 지키나**

In [ ]:
band = m3.get_prediction(test).summary_frame(alpha=0.05)
lo, hi = band["obs_ci_lower"].values, band["obs_ci_upper"].values
inside = test["발전량"].between(lo, hi)
print(f"95% 예측구간 안: {inside.mean():.1%}", f"({inside.sum()}/{len(test)})")
print("아래로 벗어남", (test["발전량"] < lo).sum(),
      "/ 위로", (test["발전량"] > hi).sum())

- `.values`: 판다스 열에서 줄 번호를 떼고 숫자 배열만 꺼냅니다. 두 표의 줄 번호가 달라도 위치대로 견주게 하려는 것입니다.
- `between(lo, hi)`: 시험 시각마다 실제 값이 구간 안에 드는지 참·거짓으로 묻습니다. `.mean()`은 참의 비율입니다.

95%라고 약속한 구간이 실제로 97.8%를 담았으니 약속을 지켰습니다. 벗어난 14시각은 아래와 위로 7개씩입니다.

**❹ 잔차 요약하기**

In [ ]:
res = test["발전량"] - m3.predict(test)  # 실제 - 예측
print(res.describe().round(2))

잔차의 평균은 +0.39MWh로 약간 위로 치우쳤고, 가장 작은 값 -7.18은 사분위수(-0.12~0.99)에서 한참 떨어져 있습니다. 혼자 떨어진 값이 있다는 신호입니다.

**❺ 가장 크게 틀린 시각 찾기**

In [ ]:
worst = test.assign(잔차=res).nsmallest(4, "잔차")
print(worst[["일시", "발전량", "일사", "적설", "잔차"]]
      .to_string(index=False, float_format="%.2f"))

- `test.assign(잔차=res)`: 잔차 열을 붙인 새 표를 만듭니다(원래 `test`는 그대로).
- `nsmallest(4, "잔차")`: 잔차가 가장 작은(예측보다 가장 적게 나온) 네 줄입니다. 반대는 `nlargest`입니다.
- `to_string(index=False, float_format="%.2f")`: 줄 번호 없이, 소수 둘째 자리까지 찍습니다.

네 시각이 모두 12월 5일 11~14시이고 적설이 1.9~2.8cm입니다. 햇빛은 왔지만 눈 덮인 패널에 닿지 못했습니다. 숫자 하나로 끝내지 않고 틀린 줄을 직접 읽으니 모델이 모르는 것(눈)이 보입니다.

**❻ 더빈-왓슨과 잔차 히스토그램**

In [ ]:
from statsmodels.stats.stattools import durbin_watson
print("더빈-왓슨:", round(durbin_watson(m3.resid), 2))
plt.figure(figsize=(5, 3.2))
plt.hist(res, bins=40, color="gray")
plt.axvline(0, color="#C8234B")
plt.xlabel("잔차 = 실제 - 예측(MWh)")
plt.ylabel("시각 수")
plt.show()

- `durbin_watson(m3.resid)`: 학습 잔차가 바로 앞 잔차와 닮았는지 잽니다. 2 근처면 닮지 않았고, 0에 가까울수록 닮았습니다.
- `plt.hist(res, bins=40)`: 잔차를 40칸으로 나눠 막대로 셉니다. `plt.axvline(0)`은 0에 세로선을 긋습니다.

**결과 읽기.** 더빈-왓슨 0.88은 이웃한 시각의 잔차가 꽤 닮았다는 뜻이라, ②의 구간은 실제보다 좁게 계산됐을 수 있습니다([표 8-3]). 히스토그램은 0 근처에 모인 종 모양이지만 오른쪽(+)으로 약간 치우쳤고, 왼쪽 끝 -7 근처의 외딴 막대가 눈 온 날입니다.

> **[개념 노트] 잔차 진단과 오차 분석**
>
> 잔차는 모델이 설명하지 못한 나머지라서, 잔차에 무늬가 남아 있으면 모델이 놓친 것이 있다는 뜻입니다. 기본 그림은 넷입니다. 예측값 대 잔차(휜 무늬는 곡선 관계, 오른쪽으로 퍼짐은 오차 크기가 일정하지 않은 이분산), 잔차 히스토그램(긴 꼬리, 떨어진 막대), 시간 순서의 잔차(같은 쪽으로 이어지면 자기상관), 독립변수 대 잔차(빠진 변수나 변환)입니다.
>
> 그다음 **오차 분석**으로 가장 크게 틀린 줄을 10개쯤 뽑아 원자료를 직접 읽고 공통점을 찾습니다. 이 장에서는 그 공통점이 "눈"이었습니다. 틀린 줄을 읽는 습관이 다음에 더할 변수를 알려 줍니다.

**[직접 해 보기]** 시험 기간에서 눈이 쌓여 있던 시각과 아닌 시각의 잔차 평균을 견주시오.

In [ ]:
# 여기에 직접 코드를 써 보십시오.

**정답 코드** (직접 해 본 뒤에 실행해 보십시오)

In [ ]:
snow = test["적설"].fillna(0) > 0
print(f"눈 쌓인 시각 {snow.sum()}개: 잔차 평균 {res[snow].mean():+.2f}"
      f" / 나머지 {res[~snow].mean():+.2f}MWh")

눈 쌓인 28시각은 예측보다 평균 0.79MWh 적게, 나머지는 0.44MWh 많이 나왔습니다. 적설을 변수로 넣을 근거입니다.

> **[자주 하는 실수] 줄 번호가 다른 두 열 견주기**
>
> 판다스는 두 열을 계산할 때 위치가 아니라 줄 번호(인덱스)로 짝을 맞춥니다. `test["발전량"]`의 줄 번호는 원래 표의 번호(예: 3968~)인데 `summary_frame`의 결과는 0부터 시작하므로, 그대로 견주면 짝이 하나도 안 맞아 결과가 엉뚱해집니다. ③에서 `.values`를 붙인 까닭입니다. 줄 번호를 0부터 다시 붙이려면 `reset_index(drop=True)`를 씁니다.

#### [실습 예제 8-9] 같은 말을 두 번 하는 변수: 분산팽창계수

일사, 기온, 습도로 회귀한 모델에 이슬점을 더하고, 변수마다 분산팽창계수(VIF)를 구해 계수가 어떻게 흔들리는지 보시오(03절 3.4).

> **[개념 노트] 다중공선성과 VIF**
>
> 독립변수끼리 거의 같은 정보를 담고 있는 것을 **다중공선성**이라고 합니다. 키를 cm와 인치로 두 번 넣은 경우를 생각하면 됩니다. 이러면 회귀는 효과를 누구 몫으로 나눌지 정하지 못해 계수가 크게 흔들립니다. 예측은 그대로여도 계수를 읽을 수 없게 됩니다.
>
> **분산팽창계수**(VIF)는 한 변수를 나머지 변수들로 회귀했을 때의 R²로 VIF = 1 ÷ (1 − R²)입니다. 다른 변수들로 90%가 설명되면 VIF는 10이고, 흔히 10을 넘으면 심하다고 봅니다. 처방은 겹치는 변수 하나를 빼거나, 릿지 회귀(6.3)를 쓰는 것입니다.

**[작은 예제로 먼저] 거의 "합"인 변수의 VIF를 손으로 구하기**

서울 날씨에 쓰기 전에, 여섯 학생의 국어·영어 점수와 "거의 총점"인 열로 VIF가 어떻게 계산되는지 단계별로 따라 해 봅니다.

**ⓐ 작은 표 만들고 상관 보기**

In [ ]:
import numpy as np
import pandas as pd
toy = pd.DataFrame({"국어": [60, 70, 80, 90, 75, 65],
                    "영어": [70, 60, 85, 80, 90, 75]})
noise = np.array([1, -1, 2, 0, -2, 1])          # 작은 흔들림
toy["총점"] = toy["국어"] + toy["영어"] + noise   # 거의 합
print(toy.corr().round(2))

- `np.array([1, -1, 2, 0, -2, 1])`: 총점에 작은 흔들림을 더해 "완전히 같지는 않게" 만듭니다. 완전히 같으면 VIF가 무한대가 되어 계산이 멈춥니다.

총점과 국어, 총점과 영어의 상관은 0.86, 0.87입니다. 두 변수씩 보면 "아주 높다"고 하기 어렵습니다.

**ⓑ 한 변수를 나머지로 회귀해 R²와 VIF 구하기**

In [ ]:
y = toy["총점"]
X = np.column_stack([np.ones(6), toy["국어"], toy["영어"]])
coef, *_ = np.linalg.lstsq(X, y, rcond=None)   # ① 총점을 나머지로 회귀
pred = X @ coef
r2 = 1 - ((y - pred) ** 2).sum() / ((y - y.mean()) ** 2).sum()
print(f"① 총점 ~ 국어 + 영어 의 R² = {r2:.4f}")
print(f"② VIF = 1 / (1 - R²) = {1 / (1 - r2):.1f}")

- `np.column_stack([np.ones(6), ...])`: 1로 채운 절편 열과 두 열을 옆으로 붙여 독립변수 표를 만듭니다.
- `np.linalg.lstsq(X, y)`: 최소제곱 계수를 구합니다. `X @ coef`는 그 계수로 낸 예측값입니다.
- VIF = 1 / (1 − R²): 나머지 변수로 설명되는 비율(R²)이 1에 가까울수록 커집니다.

**ⓒ 함수로 확인하기**

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
Xc = sm.add_constant(toy[["국어", "영어", "총점"]])
print({c: round(float(variance_inflation_factor(Xc.values, i)), 1)
       for i, c in enumerate(Xc.columns) if c != "const"})

손으로 구한 162.7과 함수의 값이 같습니다. 두 변수씩 본 상관은 0.86인데도, 국어와 영어를 "합쳐서" 보면 총점의 99.4%가 설명되어 VIF가 치솟습니다. 이제 같은 계산을 서울 날씨의 이슬점에 해 봅니다.

#### [코랩 실습 8-2] 서울 날씨에 이슬점을 더해 VIF 구하기

> 📘 책에는 문제와 핵심 결과만 있고, 코드는 이 노트북에만 있습니다.

**❶ 이슬점을 붙이고 상관부터 보기**

날씨 파일에서 이슬점 열을 더 읽어 하루 표 `day`에 붙입니다. 실습 예제 8-5에서 기온과 습도를 읽은 방법과 같습니다.

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
dew = pd.read_csv(DATA + ASOS, usecols=["일시", "이슬점온도(°C)"])
dew = dew.rename(columns={"이슬점온도(°C)": "이슬점"})
dew["일시"] = pd.to_datetime(dew["일시"])
d8 = day.merge(dew, on="일시")
print(d8[["기온", "습도", "이슬점"]].corr().round(2))

- `usecols=[...]`: 필요한 열만 읽습니다. 큰 파일일수록 빠르고 메모리도 적게 씁니다.
- `rename(columns={옛 이름: 새 이름})`: 괄호와 기호가 든 긴 열 이름을 짧게 바꿉니다. 식(formula)에 넣을 열 이름은 짧고 기호가 없어야 편합니다.
- `merge(dew, on="일시")`: 두 표를 일시가 같은 줄끼리 잇습니다. 일시의 자료형이 양쪽 모두 날짜여야 짝이 맞으므로, 앞 줄에서 `to_datetime`으로 바꿔 두었습니다.
- `.corr()`: 열끼리의 상관계수 표를 만듭니다.

기온과 이슬점의 상관이 0.91입니다. 이슬점은 기온과 습도로 계산되는 값이라 둘과 함께 움직입니다. 그런데 상관계수는 두 변수씩만 봅니다. 이슬점이 기온과 습도를 "합쳐서" 얼마나 닮았는지는 VIF로 잽니다.

**❷ 변수마다 VIF 구하기**

In [ ]:
def vif(cols):
    X = sm.add_constant(d8[cols])        # 절편 열 1을 붙인다
    return [round(float(variance_inflation_factor(X.values, i)), 1)
            for i in range(1, len(cols) + 1)]
print("일사·기온·습도 →", vif(["일사", "기온", "습도"]))
print("+ 이슬점 →", vif(["일사", "기온", "습도", "이슬점"]))

- `sm.add_constant(표)`: 값이 모두 1인 절편 열(const)을 맨 앞에 붙입니다. VIF는 절편이 있는 회귀를 전제로 하므로 빼먹으면 값이 틀립니다.
- `variance_inflation_factor(X.values, i)`: i번째 열을 나머지 열로 회귀해 VIF = 1/(1-R²)를 구합니다. 0번은 절편이라 건너뛰고 1번부터 셉니다.
- 함수 `vif(cols)`로 묶어 두니 변수 목록만 바꿔 두 번 부를 수 있습니다.

세 변수일 때는 모두 2 아래라 걱정할 것이 없습니다. 이슬점을 넣자 기온 132.9, 습도 32.9, 이슬점 186.2로 치솟았습니다. 흔히 10을 넘으면 다중공선성을 의심합니다.

**❸ 계수와 R²를 나란히 놓기**

In [ ]:
m_3 = smf.ols("발전량 ~ 일사 + 기온 + 습도", data=d8).fit()
m_dew = smf.ols("발전량 ~ 일사 + 기온 + 습도 + 이슬점", data=d8).fit()
print(pd.DataFrame({"이슬점 없이": m_3.params, "넣고": m_dew.params}).round(3))
print("R²:", round(m_3.rsquared, 4), "→", round(m_dew.rsquared, 4))

- `m.params`: 회귀 계수를 변수 이름이 붙은 열로 돌려줍니다. 두 모델의 계수를 `DataFrame`에 나란히 넣으면 이름끼리 줄이 맞고, 한쪽에만 있는 이슬점은 NaN으로 비워집니다.
- 표의 줄 순서는 판다스가 이름으로 정렬한 것이라 식에 쓴 순서와 다릅니다.

**결과 읽기.** R²는 0.9195 그대로입니다. 이슬점이 새로 알려 준 것은 없다는 뜻입니다. 그런데 기온의 계수는 -0.024에서 -0.053으로, 습도는 -0.014에서 -0.022로 크게 흔들렸습니다. 일사의 계수는 거의 그대로인데, 일사는 이슬점과 닮지 않았기 때문입니다. 예측만 할 때는 이대로 써도 되지만, "기온이 1℃ 오르면 발전량이 얼마나 줄까"처럼 계수를 읽으려면 이슬점을 빼야 합니다.

> **[자주 하는 실수] 상관계수만 보고 안심하기**
>
> 변수 셋 이상이 얽히면 두 변수씩의 상관은 낮아도 VIF는 높을 수 있습니다. 예를 들어 "총점 = 국어 + 영어 + 수학"이면 총점과 국어의 상관은 0.6쯤이지만, 총점은 나머지 셋으로 정확히 계산되므로 VIF가 무한대입니다. 상관 표로 짝을 먼저 훑고, 마지막에는 VIF로 확인합니다. 또 하나, `add_constant`를 빼고 VIF를 구하면 값이 엉뚱하게 커지거나 작아집니다.

#### [실습 예제 8-10] 릿지, 라소, 나무와 숲을 작은 예제로 + [코랩 실습 8-3]

M3의 변수에 시각을 더해 선형 회귀, 릿지, 라소, 랜덤 포레스트의 시험 성적을 같은 잣대로 견주시오(06절 6.3).

**[작은 예제로 먼저] 릿지와 라소가 계수를 줄이는 모습**

닮은 두 변수 x1, x2와 y 여섯 쌍으로, 벌점의 세기를 키울 때 계수가 어떻게 바뀌는지 직접 계산해 봅니다.

**ⓐ 표준화하고 릿지 공식으로 계수 구하기**

In [ ]:
x1 = np.array([1.0, 2, 3, 4, 5, 6])
x2 = x1 + np.array([0.1, -0.2, 0.1, 0.2, -0.1, -0.1])  # x1과 닮음
y = np.array([2.1, 3.9, 6.2, 8.1, 9.8, 12.0])
Z = np.column_stack([x1, x2])
Z = (Z - Z.mean(axis=0)) / Z.std(axis=0)   # ① 표준화
yc = y - y.mean()                       # y는 평균만 뺀다
for lam in [0, 1, 10, 100]:             # ② λ를 키우며
    w = np.linalg.solve(Z.T @ Z + lam * np.eye(2), Z.T @ yc)
    print(f"λ = {lam:>3}: 계수 {w.round(2)}, 계수 합 {w.sum():.2f}")

- `(Z - Z.mean(axis=0)) / Z.std(axis=0)`: 열마다 평균을 빼고 표준편차로 나눕니다(표준화). 단위가 다른 변수를 공정하게 벌주려는 것입니다.
- `np.linalg.solve(Z.T @ Z + lam * np.eye(2), Z.T @ yc)`: 릿지의 계수 공식 (ZᵀZ + λI)⁻¹Zᵀy를 그대로 계산합니다. λ = 0이면 보통 최소제곱과 같습니다.

λ = 0에서는 거의 같은 두 변수가 2.17과 1.21로 들쭉날쭉하게 몫을 나눴습니다. λ를 키우자 두 계수가 1.55, 0.92처럼 고르게 맞춰지며 함께 작아집니다. 다중공선성이 있을 때 릿지가 계수를 안정시킨다는 뜻이 이것입니다.

**ⓑ 라소는 계수를 0으로 만든다**

In [ ]:
from sklearn.linear_model import Lasso
for a in [0.01, 0.5, 2, 3]:             # ③ 라소의 벌점 세기
    las = Lasso(alpha=a).fit(Z, yc)
    print(f"alpha = {a:>4}: 라소 계수 {las.coef_.round(2)}")

- `Lasso(alpha=a)`: 사이킷런은 벌점의 세기 λ를 `alpha`라고 부릅니다.

alpha가 3이 되자 x2의 계수가 정확히 0이 되었습니다. 닮은 두 변수 가운데 하나만 남긴 것입니다. 릿지는 줄이기만 하고, 라소는 골라냅니다. 이제 서울 데이터의 변수 여섯 개에 같은 방법을 씁니다.

#### [코랩 실습 8-3] 서울 데이터로 선형 회귀·릿지·라소·랜덤 포레스트 견주기

> 📘 책에는 문제와 핵심 결과만 있고, 코드는 이 노트북에만 있습니다.

**❶ 세 선형 모델을 한 번에 학습하고 채점하기**

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
feats = ["일사", "기온", "운량", "습도", "온도손실", "시각"]
models = {"선형": make_pipeline(StandardScaler(), LinearRegression()),
          "릿지": make_pipeline(StandardScaler(),
                                RidgeCV(alphas=np.logspace(-3, 3, 30))),
          "라소": make_pipeline(StandardScaler(),
                                LassoCV(cv=5, random_state=0))}
for name, mdl in models.items():
    mdl.fit(train[feats], train["발전량"])
    score(name, test["발전량"], mdl.predict(test[feats]))

- 사이킷런은 독립변수 표 X(`train[feats]`)와 종속변수 y(`train["발전량"]`)를 따로 받습니다. `feats`가 X의 열 목록입니다.
- `make_pipeline(StandardScaler(), 모델)`: "표준화 → 모델"을 한 묶음으로 만듭니다. 시험 자료도 학습 자료의 평균·표준편차로 표준화되므로 정보가 새지 않습니다.
- `RidgeCV(alphas=np.logspace(-3, 3, 30))`: 0.001부터 1000까지 로그 간격으로 30개의 벌점 세기 λ(사이킷런에서는 alpha)를 시험해 가장 좋은 것을 고릅니다.
- `LassoCV(cv=5, random_state=0)`: 학습 자료를 다섯 조각으로 나눠 돌려 가며 검증(교차검증)해 λ를 고릅니다.
- 모델이 달라도 `fit( )`으로 배우고 `predict( )`로 예측하는 사용법이 같아서, for 문 하나로 모두 견줄 수 있습니다. `score( )`는 실습 예제 8-6에서 만든 채점 함수입니다.

> **[개념 노트] 표준화와 파이프라인**
>
> **표준화**는 변수마다 평균을 빼고 표준편차로 나누어 "평균 0, 표준편차 1"로 맞추는 것입니다(z-점수). 일사(0~4)와 습도(0~100)처럼 단위가 다른 변수에 같은 벌점을 주려면 먼저 표준화해야 합니다. 나무 모델은 크기를 비교만 하므로 필요 없습니다. 표준화의 평균과 표준편차는 **학습 데이터에서만** 구해야 합니다. 시험 데이터까지 넣으면 누수입니다. **파이프라인**(`make_pipeline`)은 전처리와 모델을 한 덩어리로 묶어, `fit( )` 때 학습 데이터로 구한 평균을 `predict( )` 때 그대로 쓰게 해 줍니다.

**❷ 고른 λ와 계수 들여다보기**

In [ ]:
print("릿지 λ:", round(models["릿지"][-1].alpha_, 3),
      "/ 라소 λ:", round(models["라소"][-1].alpha_, 4))
coef = pd.DataFrame({k: models[k][-1].coef_ for k in models}, index=feats)
print(coef.round(3))

- `models["릿지"][-1]`: 파이프라인의 마지막 단계, 곧 모델 자체를 꺼냅니다. `[0]`이면 표준화 단계입니다.
- `.alpha_`: 교차검증으로 고른 λ입니다. 이름 끝의 밑줄(_)은 "학습한 뒤에 생기는 값"이라는 사이킷런의 약속입니다. `coef_`도 마찬가지입니다.

릿지는 λ 0.788, 라소는 λ 0.0039를 골랐습니다. 둘 다 작은 값이라 계수가 선형 회귀와 거의 같고, 라소도 0으로 만든 계수가 없습니다. 변수가 여섯 개뿐이고 데이터가 수천 줄이라 벌점이 할 일이 거의 없었던 것입니다. 표준화한 변수의 계수라서 크기를 서로 견줄 수 있고, 일사(3.677)가 가장 크고 시각(0.054)이 가장 작습니다.

> **[개념 노트] 과적합과 과소적합**
>
> **과적합**은 모델이 학습 데이터의 우연한 흔들림까지 외워, 학습 성적은 좋은데 새 데이터에서 성적이 떨어지는 것입니다. 기출문제 답만 외운 학생과 같습니다. **과소적합**은 모델이 너무 단순해 학습 데이터도 잘 맞히지 못하는 것입니다(일사 하나뿐인 M1). 학습 성적과 시험 성적을 나란히 놓고, 둘 다 나쁘면 변수나 모델을 더하고, 학습만 좋으면 모델을 단순하게 하거나 벌점(릿지·라소)을 주거나 데이터를 더 모읍니다. 이 균형을 **편향-분산 트레이드오프**라고 합니다.

> **[개념 노트] 검증 데이터와 교차검증**
>
> λ 같은 하이퍼파라미터를 시험 성적을 보며 고르면 시험 데이터를 학습에 쓴 셈입니다. 그래서 학습 데이터 안에서 **검증 데이터**를 떼어 고르고, 시험 데이터는 맨 마지막에 한 번만 씁니다. **k-겹 교차검증**은 학습 데이터를 k조각으로 나누어 한 조각씩 돌아가며 검증에 쓰고 성적을 평균합니다. `LassoCV(cv=5)`가 5겹 교차검증으로 λ를 고른 것입니다. 시간 순서가 있는 데이터는 앞으로 배우고 바로 뒤로 검증하는 `TimeSeriesSplit`을 씁니다.

**[작은 예제로 먼저] 나무 한 그루가 자르는 법, 숲이 평균하는 법**

랜덤 포레스트는 "회귀나무"를 여러 그루 키워 평균합니다. 일사와 발전량 여덟 쌍으로 나무 한 그루가 자를 곳을 고르는 과정을 손으로 따라 합니다.

**ⓐ 자를 곳마다 SSE 합 재기**

In [ ]:
sun = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0])  # 일사
gen = np.array([1.0, 2.0, 2.5, 5.0, 6.0, 6.5, 9.0, 9.5])   # 발전량
def sse(v):                              # 평균에서 잰 제곱합
    return ((v - v.mean()) ** 2).sum()
for cut in [0.75, 1.25, 1.75, 2.25, 2.75, 3.25, 3.75]:  # ①
    left, right = gen[sun < cut], gen[sun >= cut]
    print(f"일사 < {cut}: 왼쪽 {len(left)}개, 오른쪽 {len(right)}개,"
          f" SSE 합 {sse(left) + sse(right):6.2f}")

- `gen[sun < cut]`: 일사가 기준보다 작은 줄의 발전량만 고릅니다(왼쪽 무리). 나머지가 오른쪽 무리입니다.
- `sse(v)`: 무리의 평균에서 잰 제곱합입니다. 무리 안이 고를수록 작습니다.

**ⓑ 가장 작은 곳에서 자르고 잎마다 평균으로 예측하기**

In [ ]:
cut = 1.75                       # ② SSE 합이 가장 작은 곳
left, right = gen[sun < cut], gen[sun >= cut]
print(f"왼쪽 잎 {left.mean():.2f}, 오른쪽 잎 {right.mean():.2f}")
from sklearn.tree import DecisionTreeRegressor
X1 = sun.reshape(-1, 1)                    # 한 열짜리 표로
stump = DecisionTreeRegressor(max_depth=1).fit(X1, gen)  # ③
print("사이킷런이 고른 기준:", stump.tree_.threshold[0],
      "/ 일사 3.2의 예측:", stump.predict([[3.2]])[0])

- `DecisionTreeRegressor(max_depth=1)`: 한 번만 자르는 나무(그루터기)입니다. `tree_.threshold[0]`이 고른 기준입니다.
- `reshape(-1, 1)`: 사이킷런은 X를 표(2차원)로 받으므로 한 줄짜리 배열을 세로 한 열로 바꿉니다.

SSE 합이 가장 작은 일사 1.75에서 잘랐고, 사이킷런도 같은 곳을 골랐습니다. 나무의 예측은 잎(마지막 무리)의 평균입니다. 일사 3.2이면 오른쪽 잎의 평균 7.2를 냅니다.

**ⓒ 다시 뽑아 여러 그루, 그리고 평균**

In [ ]:
rng = np.random.default_rng(0)
preds = []
for k in range(3):                    # ④ 나무 세 그루
    idx = rng.integers(0, 8, 8)       # 중복 허락해 8개 뽑기
    t = DecisionTreeRegressor(max_depth=1).fit(X1[idx], gen[idx])
    p = t.predict([[3.2]])[0]; preds.append(p)
    th = t.tree_.threshold[0]
    print(f"나무 {k + 1}: 뽑힌 줄 {sorted(idx.tolist())},"
          f" 기준 {th:.2f}, 예측 {p:.2f}")
print(f"숲의 예측(세 나무의 평균): {np.mean(preds):.2f}")

- `rng.integers(0, 8, 8)`: 0~7번 줄을 중복을 허락해 8번 뽑습니다(부트스트랩). 나무마다 조금씩 다른 데이터를 봅니다.

나무마다 뽑힌 줄이 달라 자른 곳(2.00, 3.25)과 예측(7.17, 5.40, 7.25)이 다릅니다. 숲은 이를 평균해 6.61을 냅니다. 한 그루의 들쭉날쭉함을 평균으로 누그러뜨리는 것이 랜덤 포레스트입니다. 아래에서는 서울 데이터로 나무 300그루를 키웁니다.

#### [코랩 실습 8-3 이어서] 랜덤 포레스트

> 📘 책에는 문제와 핵심 결과만 있고, 코드는 이 노트북에만 있습니다.

**❸ 랜덤 포레스트와 학습·시험 성적의 틈**

In [ ]:
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                           random_state=0, n_jobs=-1)
rf.fit(train[feats], train["발전량"])
score("랜덤 포레스트", test["발전량"], rf.predict(test[feats]))
score("(학습 성적)", train["발전량"], rf.predict(train[feats]))
imp = pd.Series(rf.feature_importances_, index=feats)
print(imp.sort_values(ascending=False).round(3).to_dict())

- `n_estimators=300`: 나무 300그루의 예측을 평균합니다. 많을수록 안정되지만 느려집니다.
- `min_samples_leaf=5`: 잎(마지막 칸)마다 최소 5시각을 남겨, 나무가 한두 시각에 맞춰 외우지 못하게 합니다.
- `random_state=0`: 무작위로 뽑는 과정을 고정해 누가 돌려도 같은 결과가 나오게 합니다. `n_jobs=-1`은 컴퓨터의 모든 코어를 씁니다.
- 학습 자료로도 채점해 시험 성적과 견줍니다. `feature_importances_`는 변수마다 예측 오차를 줄인 몫(합이 1)입니다.

**결과 읽기.** 랜덤 포레스트가 시험 MAE 0.47로 선형 모델(0.67)보다 좋았습니다. 일사와 발전량의 관계가 완전한 직선이 아니고, 시각과 온도에 따라 꺾이는 부분을 나무가 잡아낸 것입니다. 다만 학습 성적(0.35)과 시험 성적(0.47) 사이에 틈이 있고, 이 틈이 과적합의 크기입니다. 중요도는 일사가 0.947로 거의 전부입니다. 계수와 달리 중요도에는 부호가 없어서 "늘면 줄어든다"는 방향은 알려 주지 않습니다.

> **[개념 노트] 랜덤 포레스트 한눈에**
>
> **결정 나무**는 "일사가 1.5보다 큰가?" 같은 질문을 차례로 던져 데이터를 나누고, 마지막 칸의 평균으로 예측합니다. **랜덤 포레스트**는 데이터와 변수를 조금씩 무작위로 골라 나무를 수백 그루 기르고 그 예측을 평균해, 나무 하나의 실수를 서로 지웁니다. 휘어진 관계와 상호작용을 스스로 찾는 대신, "일사 1이 늘면 4MWh"처럼 한 문장으로 읽을 수는 없습니다. 나무 모델은 10장에서 자세히 다룹니다.

> **[자주 하는 실수] 시험 자료로 λ 고르기**
>
> 시험 자료의 성적을 보며 λ나 나무 수를 이리저리 바꾸면, 시험 자료가 사실상 학습에 쓰여 성적이 실제보다 좋게 나옵니다. 하이퍼파라미터는 `RidgeCV`, `LassoCV`처럼 학습 자료 안의 교차검증으로 고르고, 시험 자료는 마지막에 한 번만 씁니다. 표준화도 마찬가지로 전체 자료가 아니라 학습 자료의 평균으로 해야 하며, 파이프라인이 이를 대신 지켜 줍니다.

> **핵심 3줄**
>
> 판다스로 고르고(조건), 잇고(merge), 묶고(groupby), 빈칸의 뜻을 따진 뒤에 모델을 만듭니다.
>
> 학습과 시험은 날짜로 자르고, 채점 함수 하나로 기준선과 모델을 같은 잣대로 재며, 학습·시험 성적의 틈으로 과적합을 봅니다.
>
> 숫자 하나로 끝내지 말고 잔차를 그리고 가장 크게 틀린 줄을 읽으면, 모델이 모르는 것(눈)과 다음에 더할 변수가 보입니다.

## [코랩 실습 8-4] 원자료를 직접 받아 정리하기

책 [실습 예제 8-5] 끝의 [코랩 실습 8-4] 상자를 코드로 해 봅니다. 포털에서 받은 원자료가 DATA 폴더에 없으면 아무 일도 하지 않고 넘어갑니다.

In [ ]:
# ☆ 원자료에서 정리본 만들기: 공공데이터포털·기상자료개방포털에서 받은 파일이 DATA 폴더에 있을 때만 돈다.
import glob
raw_asos = sorted(glob.glob(DATA + "OBS_ASOS_TIM_*.csv"))      # 기상자료개방포털: 한 번에 1년쯤씩 받은 파일들
if raw_asos:
    parts = [pd.read_csv(f, encoding="cp949") for f in raw_asos]  # 원자료는 한글 윈도 인코딩(cp949)
    asos = pd.concat(parts).drop_duplicates("일시").sort_values("일시")
    asos.to_csv(DATA + "asos_내가정리한_서울108.csv", index=False)  # UTF-8로 저장
    print("ASOS 파일", len(raw_asos), "개 →", len(asos), "시각", asos["일시"].iloc[0], "~", asos["일시"].iloc[-1])
    print("일사 빈칸(밤 포함):", asos["일사(MJ/m2)"].isna().sum(), "/ 강수량 빈칸(비가 안 온 시각):", asos["강수량(mm)"].isna().sum())
else:
    print("기상청 원자료(OBS_ASOS_TIM_*.csv)가 없어 건너뜁니다.")

raw_kpx = sorted(glob.glob(DATA + "한국전력거래소_*태양광*풍력*.csv"))  # 공공데이터포털에서 받은 파일
if raw_kpx:
    sol = pd.read_csv(raw_kpx[0], encoding="cp949")
    sol.columns = ["거래일", "거래시간", "지역", "연료원", "전력거래량(MWh)"]
    sol.to_csv(DATA + "kpx_내가정리한_태양광풍력.csv", index=False)
    print("한국전력거래소 파일:", sol.shape, "/ 거래시간", sol["거래시간"].min(), "~", sol["거래시간"].max())
else:
    print("한국전력거래소 원자료(한국전력거래소_*태양광*풍력*.csv)가 없어 건너뜁니다.")

raw_dem = sorted(glob.glob(DATA + "한국전력거래소_시간별*전력수요량*.csv"))  # 미니 프로젝트: 해마다 한 파일
if raw_dem:
    parts = []
    for f in raw_dem:
        t = pd.read_csv(f, encoding="cp949")
        t.columns = ["날짜"] + [f"{h}시" for h in range(1, 25)]   # 해마다 열 이름을 똑같이
        parts.append(t)
    dem_all = pd.concat(parts).dropna(subset=["날짜"])            # 내용 없는 줄을 버린다
    dem_all = dem_all.drop_duplicates("날짜").sort_values("날짜")
    dem_all.to_csv(DATA + "kpx_내가정리한_전력수요.csv", index=False)
    print("전력수요 파일", len(raw_dem), "개 →", dem_all.shape, dem_all["날짜"].iloc[0], "~", dem_all["날짜"].iloc[-1])
else:
    print("한국전력거래소 원자료(한국전력거래소_시간별*전력수요량*.csv)가 없어 건너뜁니다.")

## [미니 프로젝트 8] 추우면 난방, 더우면 냉방: 전력 수요의 U자

**질문.** 전국의 하루 전력 수요는 날씨에 얼마나 끌려다닐까요? 추운 날은 난방으로, 더운 날은 냉방으로 전기를 많이 씁니다. 그렇다면 기온과 전력 수요 사이에 직선 하나를 그어도 될까요?

**데이터.** 두 파일을 씁니다. 하나는 한국전력거래소의 "시간별 전국 전력수요량"(2021~2025년)입니다. 한 행이 하루이고, 날짜와 1시~24시의 수요(MW)로 25열입니다. 다른 하나는 03절의 기상청 서울 관측 자료입니다. 서울의 하루 평균 기온을 전국 날씨의 대표로 씁니다. 두 자료가 겹치는 2023~2025년 1,096일이 분석 대상입니다. 수업 자료의 정리본(kpx_시간별_전국_전력수요량_2021_2025.csv)을 DATA 폴더에 두고 시작합니다.


**1단계: 데이터 신원 확인**([실습 예제 8-5] 응용). 수요 파일의 빈 줄을 확인하고 하루 수요(GWh)를 만드시오. 서울의 하루 평균 기온을 붙이시오.

실습 예제 8-6의 채점 함수 `score( )`에 단위(`unit`)를 고를 수 있게 한 칸을 더합니다. 전력 수요는 GWh로 채점합니다.

In [ ]:
def score(name, y_true, y_pred, unit="MWh"):   # 실습 8-5의 채점 함수 + 단위
    mae = mean_absolute_error(y_true, y_pred)
    print(f"{name:8s} MAE {mae:6.2f}{unit}  R² {r2_score(y_true, y_pred):6.3f}")

In [ ]:
DEMAND = "kpx_시간별_전국_전력수요량_2021_2025.csv"  # 수업 자료로 받은 정리본
dem = pd.read_csv(DATA + DEMAND)
print(dem.shape, "/ 날짜가 빈 줄", dem["날짜"].isna().sum())
dem = dem.dropna(subset=["날짜"])
dem["날짜"] = pd.to_datetime(dem["날짜"])
dem["수요"] = dem.drop(columns="날짜").sum(axis=1) / 1000   # 24시간 MW의 합 → 하루 GWh

temp = pd.read_csv(DATA + ASOS, usecols=["일시", "기온(°C)"])
temp["날짜"] = pd.to_datetime(temp["일시"]).dt.normalize()
temp = temp.groupby("날짜")["기온(°C)"].mean().rename("기온").reset_index()

mp = dem[["날짜", "수요"]].merge(temp, on="날짜")       # 두 자료가 겹치는 날만
print(len(mp), "일:", mp["날짜"].min().date(), "~", mp["날짜"].max().date())
print(mp[["수요", "기온"]].describe().round(1).loc[["mean", "min", "max"]].to_string())

수요 파일 맨 끝에 날짜까지 빈 줄이 하나 있어 버렸습니다. 24시간의 MW를 더해 1,000으로 나누면 하루 GWh입니다. 하루 수요는 1,089~1,997GWh, 기온은 −13.8~32.7℃ 사이입니다.

**2단계: 기준선과 직선**([실습 예제 8-6] 응용). 2023~2024년으로 배우고 2025년으로 시험하시오. 평균으로 찍기와 직선 회귀를 견주고, 산점도를 그리시오.

In [ ]:
mp["연도"] = mp["날짜"].dt.year
tr, te = mp[mp["연도"] < 2025], mp[mp["연도"] == 2025]   # 2023~2024년으로 배우고 2025년으로 시험
line = smf.ols("수요 ~ 기온", data=tr).fit()
print(f"직선: 수요 = {line.params['Intercept']:.1f} {line.params['기온']:+.2f} × 기온,  학습 R² {line.rsquared:.3f}")
print(f"상관계수 {mp['수요'].corr(mp['기온']):.3f}")
score("평균으로 찍기", te["수요"], np.full(len(te), tr["수요"].mean()), "GWh")
score("직선", te["수요"], line.predict(te), "GWh")

plt.figure(figsize=(5, 3.6))
plt.scatter(mp["기온"], mp["수요"], s=6, alpha=0.4, color="gray")
plt.xlabel("서울 하루 평균 기온(°C)")
plt.ylabel("전국 하루 전력 수요(GWh)")
plt.show()

상관계수가 0.004, 직선의 학습 R²가 0.002입니다. 직선으로 보면 "기온과 수요는 아무 관계가 없다"입니다. 하지만 산점도는 뚜렷한 U자입니다. 추운 쪽의 오르막과 더운 쪽의 오르막이 직선 하나 안에서 서로 지워진 것입니다. 상관계수가 0이라고 관계가 없는 것은 아닙니다.

**3단계: 제곱항과 쉬는 날 더하기**. 기온의 제곱을 변수로 더해 U자를 그리고, 주말과 공휴일을 뜻하는 "쉬는날" 더미 변수도 더하시오.

In [ ]:
import holidays
kr = holidays.KR(years=[2023, 2024, 2025])
mp["쉬는날"] = ((mp["날짜"].dt.dayofweek >= 5) | mp["날짜"].isin(pd.to_datetime(list(kr)))).astype(int)
tr, te = mp[mp["연도"] < 2025], mp[mp["연도"] == 2025]
u = smf.ols("수요 ~ 기온 + I(기온 ** 2)", data=tr).fit()
u2 = smf.ols("수요 ~ 기온 + I(기온 ** 2) + 쉬는날", data=tr).fit()
print(f"U자: 학습 R² {u.rsquared:.3f}, 수요가 가장 적은 기온 {-u.params['기온'] / (2 * u.params['I(기온 ** 2)']):.1f}℃")
print(f"U자 + 쉬는날: 학습 R² {u2.rsquared:.3f}, 쉬는 날 계수 {u2.params['쉬는날']:.1f}GWh")
score("U자", te["수요"], u.predict(te), "GWh")
score("U자 + 쉬는날", te["수요"], u2.predict(te), "GWh")

t = np.linspace(mp["기온"].min(), mp["기온"].max(), 100)
plt.figure(figsize=(5, 3.6))
plt.scatter(tr["기온"], tr["수요"], s=6, alpha=0.4, color="gray")
plt.plot(t, u.predict(pd.DataFrame({"기온": t})), color="#C8234B")
plt.xlabel("서울 하루 평균 기온(°C)")
plt.ylabel("전국 하루 전력 수요(GWh)")
plt.show()

기온의 제곱을 넣어도 여전히 "직선 회귀"입니다. 계수에 대해 일차식이기 때문입니다. U자 곡선은 2025년 시험 R² 0.446, MAE 112.66GWh로 직선(156.81GWh)보다 훨씬 낫습니다. 쉬는 날을 더하자 MAE가 76.20GWh로 줄었습니다. 쉬는 날에는 같은 기온에서 225.8GWh를 덜 씁니다. 공장과 사무실이 쉬기 때문입니다.

**4단계: 가장 크게 틀린 날 읽기.** 2025년에 예측보다 수요가 가장 적었던 날 다섯을 찾고, 그날이 어떤 날이었는지 쓰시오.

In [ ]:
te = te.assign(잔차=te["수요"] - u2.predict(te))
print(te.nsmallest(5, "잔차")[["날짜", "수요", "기온", "쉬는날", "잔차"]].to_string(index=False, float_format="%.1f"))

다섯 날 가운데 셋이 설 연휴(1월 28~30일)입니다. 주말 하루와 설 연휴 사흘은 같은 "쉬는날"이 아니었습니다. 5월 2일은 근로자의 날과 어린이날 연휴 사이에 낀 금요일, 12월 26일은 연말입니다. 달력에는 평일이지만 많은 사람이 쉰 날입니다.

**5단계: 한계 쓰기.** 이 모델이 아직 모르는 것을 [표 8-3]처럼 세 가지 이상 쓰시오. 12장의 미니 프로젝트에서는 같은 수요를 "어제와 지난주의 수요"로 예측합니다. 기온과 지난주 값 가운데 무엇이 더 쓸모 있을지도 생각해 보십시오.

**제출물.** 그림 2장(기온과 수요의 산점도와 U자 곡선, 잔차가 큰 날 표시), 성적표 1개(평균, 직선, U자, U자 + 쉬는날의 시험 MAE와 R²), 결론 5문장 이내.

**[표 8-4] 미니 프로젝트 채점 기준**

| 항목(비중) | 상 | 중 | 하 |
|---|---|---|---|
| 데이터 점검(20%) | 빈 줄, 단위(MW → GWh), 두 자료가 겹치는 기간을 모두 밝힘 | 빈 줄만 처리함 | 점검 없이 바로 모델을 만듦 |
| 기준선 비교(20%) | 해로 나누고 평균 찍기와 견줌 | 기준선은 있으나 무작위로 나눔 | 기준선 없이 R²만 씀 |
| 기법 적용(30%) | 제곱항과 더미 변수를 넣고 수요가 가장 적은 기온을 구함 | 제곱항만 넣음 | 직선만 쓰거나 코드가 돌지 않음 |
| 해석과 한계(30%) | 상관계수 0의 함정과 연휴의 잔차를 설명하고 한계를 셋 이상 씀 | 한계를 한두 가지 씀 | 숫자만 나열함 |

## ✏️ [연습문제 8-5] 시험 기간 바꾸기

[실습 예제 8-7]에서 나누는 날을 2025년 7월 1일로 바꾸어(1~6월로 학습, 7~12월로 시험) M1과 M3의 시험 MAE와 R²를 구하시오. 원래(1~10월 학습, 11~12월 시험)와 견주어 설명하시오.

In [ ]:
# 여기에 직접 코드를 써 보십시오.

### 정답 코드

직접 해 본 뒤에 실행해 보십시오.

In [ ]:
CUT2 = "2025-07-01"                                # 앞 6달로 배우고 뒤 6달로 시험
train2, test2 = day[day["일시"] < CUT2], day[day["일시"] >= CUT2]
for name in ["M1", "M3"]:
    fit = smf.ols(forms[name], data=train2).fit()
    score(name, test2["발전량"], fit.predict(test2))